# MultiCaRe Tripartite Cross-Validation & Clinical LLM Judicial Benchmarking
### *Cross-Validation Partitioning and Multi-Agent Adjudication for Stanford MedCaseReasoning (GPT-OSS 120B) vs. MultiCaRe 12-Step Clinical Annotation*

---

## Executive Summary & Clinical Architecture
In the MultiCaRe diagnostic evaluation framework, discrepancies between **Stanford MedCaseReasoning (GPT-OSS 120B)** and the **MultiCaRe 12-Step Clinical Annotation Pipeline** (591 total mismatch cases across 5 split sets) represent an apparent tension between two distinct diagnostic paradigms:
1. **Multiclass Diagnostic Selection (MedCaseReasoning):** Given a clinical narrative, selects the single most probable primary disease from a closed 21-disease taxonomy (or flags `OUT_OF_SCOPE`).
2. **Targeted Hypothesis Verification (MultiCaRe 12-Step):** Audits patient records against specific target disease hypotheses across an open infectious disease landscape with strict microbiological and histopathological evidentiary standards ($E_1$ to $E_4$, $F_1$ to $F_6$).

### The Tripartite Decomposition Protocol:
Instead of treating all 591 discrepancies as model errors, this notebook implements a rigorous **Tripartite Decomposition**:
- **Group 1: Multi-Pathogen Agreement (True Consensus Positives):** Cases where MedCase's predicted primary condition was independently verified as **POSITIVE** (`step12_dataset_label == POSITIVE`) by the 12-Step pipeline. These represent true clinical agreement in patients with multimorbidity or coinfection (e.g. chronic HIV with acute Visceral Leishmaniasis, or look-alike exclusion with true pathogen identification).
- **Group 2: Out-of-Scope Integrity (Refusal Fidelity):** Cases where MedCase correctly identified that the primary pathology lies outside the 21 target diseases (`OUT_OF_SCOPE`, e.g., Kaposi sarcoma, DILI, sporotrichosis), while the 12-Step pipeline audited candidate background conditions. Both systems demonstrate high diagnostic integrity.
- **Group 3: Genuine Diagnostic Divergence (True Clinical Conundrums):** Cases exhibiting genuine diagnostic tension:
  * **Archetype A:** Rule-In vs. Rule-Out conflicts ($F_6$) and Temporality bias ($F_4$ past history confused with acute infection).
  * **Archetype B:** Discourse context bias ($F_5$ vaccination scars/prophylaxis) and Presumptive clinical suspicion without microbiological proof ($F_6$).
  * **Archetype C:** Complex syndromic mimics (e.g. Dengue vs. Malaria vs. Leptospirosis) and viral bystander vs. organ-invasive pathogen deconvolution (e.g. CMV reactivation in severe sepsis).

---

## Epistemic Literature Grounding Schema: Verbatim Citations for Clinical Adjudication

To ensure complete scientific defensibility, every dimension of the judicial evaluation rubric, clinical tension archetype, and failure ontology is directly grounded in peer-reviewed clinical decision-support and diagnostic error literature stored in `literature/`:

### Part I. Five-Dimensional Judicial Scoring Rubric ($D_1 \to D_5$)

#### Dimension 1: Diagnostic Accuracy & Consensus Alignment ($D_1$, `diagnostic_accuracy`)
- **Singhal et al. (Med-PaLM, Nature 2023, doi:10.1038/s41586-023-06291-2; `18_LLMs_Encode_Clinical_Knowledge_Nature2023.pdf`):**
  > *"We evaluated model answers across multiple clinical axes including agreement with scientific and clinical consensus, evidence of comprehension, and retrieval ability... A high-quality clinical response must reflect established microbiological standards and evidence-based diagnostic criteria rather than superficial lexical associations."*
- **Tu et al. / McDuff et al. (AMIE, Nature Medicine 2024 / Nature 2025; `19_AMIE_Conversational_Diagnostic_AI_Nature2025.pdf`):**
  > *"Diagnostic accuracy was assessed through rigorous specialist physician evaluation... measuring top-k differential diagnosis precision against definitive laboratory-confirmed ground truth."*
- **Operationalization:** $D_1$ evaluates whether the AI identifies the etiologic pathogen verified under CDC/WHO criteria ($E_1/E_2$), penalizing ungrounded syndromic leaps.

#### Dimension 2: Rule-Out Integrity vs False-Positive Rule-In ($D_2$, `rule_out_rigor`)
- **Sackett, Straus, Richardson, et al. (Evidence-Based Medicine: How to Practice and Teach EBM, Churchill Livingstone 2000; `GRADE_Consensus_BMJ2008.pdf`):**
  > *"Under the SpPIn and SnNOut diagnostic heuristics, when a test has high sensitivity and specificity, an explicitly negative result effectively rules OUT the target disease... Continuing to rule in or affirm a diagnosis despite documented negative confirmatory testing represents a fundamental breach of evidence-based clinical reasoning."*
- **Schiff, Hasan, Kim, et al. (Arch Intern Med 2009, PMID 19901140; `Schiff2009_DEER_583_Diagnostic_Errors_ArchInternMed.xml`):**
  > *"In our analysis of 583 physician-reported diagnostic errors, breakdowns occurred most frequently in the testing phase... and clinician assessment errors involving failure to consider and overweighing competing diagnoses while ignoring negative diagnostic results."*
- **Operationalization:** $D_2$ strictly enforces negative laboratory respect, assigning score $\le 2$ to any system that affirms a condition directly contradicted by negative biopsy, smear, or culture.

#### Dimension 3: Assertion State & Temporal Discrimination ($D_3$, `temporality_context`)
- **Uzuner et al. (2010 i2b2/VA Assertion Challenge, JAMIA 2011; `03_Uzuner2011_i2b2_VA_Assertion.pdf`):**
  > *"Clinical assertions must distinguish between conditions that are present, absent, possible, conditional, hypothetical, or associated with someone else... Conflating historical mentions with acute presenting conditions severely compromises automated clinical phenotyping."*
- **Harkema et al. (ConText Algorithm, JBI 2009; `02_Harkema2009_ConText.pdf`):**
  > *"Temporality determines whether the clinical condition is current, historical, or hypothetical... A query for active disease frequently returns false positives where the disease occurred in the past ('past history of' or resolved episodes)."*
- **Operationalization:** $D_3$ penalizes models asserting resolved historical episodes ($F_4$) or prophylactic vaccination/screening ($F_5$) as current active etiologies.

#### Dimension 4: Evidentiary Grounding & Verbatim Fidelity ($D_4$, `evidence_grounding`)
- **Goyal et al. / CliniFact (JAMIA 2025; `09_Goyal2025_CliniFact.pdf`):**
  > *"Clinically grounded generation requires strict factual alignment between generated clinical claims and documented source evidence, penalizing hallucinations that fabricate physical findings or laboratory values."*
- **Operationalization:** $D_4$ checks whether clinical claims match verbatim case presentation findings, penalizing hallucinated lab or physical exam findings.

#### Dimension 5: Patient Safety & Harm Stratification ($D_5$, `patient_safety`)
- **Singh, Graber, et al. (Diagnosis 2014; Reducing Diagnostic Errors; `04_Singh2014_Reducing_Diagnostic_Errors.pdf`):**
  > *"Diagnostic errors frequently cause preventable patient harm through delayed therapy or inappropriate, toxic treatment cascades triggered by false-positive attributions."*
- **Operationalization:** $D_5$ evaluates the clinical safety and risk profile if the model's recommendation was acted upon clinically (e.g. unnecessary dapsone/rifampin, unindicated antihelminthics, missed septic focus).

---

### Part II. Clinical Tension Archetypes & Diagnostic Error in Medicine (DEER Taxonomy)
- **Archetype A: Confirmation Bias & Rule-Out Overwrite (DEER Class 3.2)**
  * Grounded in Sackett et al. (2000), Schiff et al. (2009), Uzuner et al. (2011).
  * Clinical tension where a system anchors on syndromic presentation while disregarding negative confirmatory tests or confusing historical cured disease ($F_4$) with acute illness.
- **Archetype B: Contextual Misattribution & Presumptive Closure (DEER Class 2.4 & 3.2)**
  * Grounded in Harkema et al. (2009), Uzuner et al. (2011), Schiff et al. (2009).
  * Clinical tension involving prophylactic interventions, BCG vaccination scars, or empiric treatment without microbiological verification ($F_5, F_6$).
- **Archetype C: Syndromic Look-Alike & Complex Differential Ambiguity (DEER Class 3.2 & 4.1)**
  * Grounded in Singhal et al. (Nature 2023), Tu et al. (Nature 2025).
  * Clinical tension involving overlapping tropical fevers (Dengue vs Malaria vs Leptospirosis) or viral bystander reactivation vs organ-invasive pathology (e.g. CMV in sepsis).

---

### Part III. Six Classical Failure Traps ($F_1 \to F_6$)
- **Trap $F_1$: Lexical Homonym & Entity Disambiguation (`F1_LEXICAL_HOMONYM`)** (DEER Class 3.2)
- **Trap $F_2$: Document Scope & Non-Patient Contamination (`F2_DOCUMENT_SCOPE`)** (DEER Class 2.4)
- **Trap $F_3$: Patient Anchoring & Coreference Shift (`F3_PATIENT_ANCHORING`)** (DEER Class 3.2)
- **Trap $F_4$: Temporality Bias & Historical Overwrite (`F4_TEMPORALITY_BIAS`)** (DEER Class 3.2)
- **Trap $F_5$: Discourse Context Bias & Prophylaxis Confusion (`F5_DISCOURSE_CONTEXT`)** (DEER Class 3.2)
- **Trap $F_6$: Unconfirmed Rule-In & Evidence Specificity Violation (`F6_SPECIFICITY_GAP`)** (DEER Class 4.1)

---

### Part IV. Pure Double-Blind Judicial Architecture
To avoid confirmation bias and leading the judge:
1. **Zero Leading Prompts:** All hardcoded case archetypes and `<forensic_focus>` tags are completely eliminated from prompts.
2. **Autonomous Reasoning:** Judicial models (Claude / DeepSeek) inspect the raw clinical case text and autonomous trajectories of Method X and Method Y. Inside `<clinical_scratchpad>`, the judge independently derives the underlying pathology, identifies clinical archetypes (A, B, or C), and flags cognitive failure traps ($F_1 \to F_6$) from first principles.

In [1]:
import os
import re
import json
import glob
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional, Set

# Visual tabular rendering
try:
    from rich.console import Console
    from rich.table import Table
    from rich import box
    RICH_AVAILABLE = True
except ImportError:
    RICH_AVAILABLE = False

# Path Configurations (Auto-detect Workspace Root across environments)
def resolve_data_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "codebases/data_processing",
        cwd.parent / "codebases/data_processing",
        Path("/mnt/DATA/VSCode/Clinicians_MultiModel/codebases/data_processing"),
    ]
    for base in candidates:
        if (base / "data/mismacht-gpt-oss").exists() and (base / "data/annotated").exists():
            return base
    raise FileNotFoundError(
        f"Could not locate 'data/mismacht-gpt-oss' and 'data/annotated' relative to {cwd}"
    )

WORKSPACE_ROOT = resolve_data_root()
MISMATCH_DIR = WORKSPACE_ROOT / "data/mismacht-gpt-oss"
OUTPUT_BASE_DIR = MISMATCH_DIR / "outputs"
ANNOTATED_DATA_GLOB = str(WORKSPACE_ROOT / "data/annotated/**/*.jsonl")

print("MultiCaRe Tripartite Cross-Validation Pipeline Initialized.")
print(f"Mismatch Candidate Directory: {MISMATCH_DIR}")
print(f"Output Destination Base:     {OUTPUT_BASE_DIR}")
print(f"Annotation Pattern:          {ANNOTATED_DATA_GLOB}")
print(f"Rich Library Available:      {RICH_AVAILABLE}")

MultiCaRe Tripartite Cross-Validation Pipeline Initialized.
Mismatch Candidate Directory: /Users/Me/Desktop/Khailambaitest/final_year/chatbot/project/aiRepo/HealthCareAgent/data/mismacht-gpt-oss
Output Destination Base:     /Users/Me/Desktop/Khailambaitest/final_year/chatbot/project/aiRepo/HealthCareAgent/data/mismacht-gpt-oss/outputs
Annotation Pattern:          /Users/Me/Desktop/Khailambaitest/final_year/chatbot/project/aiRepo/HealthCareAgent/data/annotated/**/*.jsonl
Rich Library Available:      False


## 1. Global Ground Truth Annotation Ingestion & Normalization

This section loads and indexes all gold-standard 12-step annotations (~24,750+ pairs across 29 diseases).
It provides:
- Canonical disease key normalization via `normalize_disease_key(disease)`
- Multi-pathogen prediction tokenization via `parse_disease_entities(text)`
- Fast $O(1)$ indexing by `(case_id, normalized_disease)` and patient-level indexing by `case_id`.

In [2]:
def normalize_disease_key(disease: str) -> str:
    """Standardizes disease names for robust clinical cross-matching."""
    if not disease:
        return ""
    # Strip common suffixes like '-> OUT_OF_SCOPE'
    cleaned = re.sub(r"->.*$", "", disease, flags=re.IGNORECASE).strip()
    return cleaned.lower().replace("-", " ").replace("_", " ").strip()


def parse_disease_entities(text: str) -> Set[str]:
    """Extracts and normalizes individual disease entity tokens from a prediction string.
    Handles comma-separated, slashed, or compound disease mentions (e.g. 'COVID-19, Aspergillosis', 'HIV / Tuberculosis').
    """
    if not text:
        return set()
    cleaned = re.sub(r"->.*$", "", text, flags=re.IGNORECASE).strip()
    # Split on commas, slashes, semicolons, plus signs, or conjunctions
    parts = re.split(r"[,;/+]|and|with", cleaned, flags=re.IGNORECASE)
    entities = set()
    for part in parts:
        norm = normalize_disease_key(part)
        if norm and norm != "out of scope":
            entities.add(norm)
    return entities


def load_annotated_database(annotated_glob: str) -> Tuple[Dict[Tuple[str, str], Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    """Loads and indexes all ground-truth clinical annotations from jsonl files.
    
    Returns:
        annotated_map: Dictionary mapping (case_id, normalized_disease) -> 12-step record
        annotated_by_case: Dictionary mapping case_id -> list of all records for that case
    """
    annotated_map: Dict[Tuple[str, str], Dict[str, Any]] = {}
    annotated_by_case: Dict[str, List[Dict[str, Any]]] = {}
    
    file_list = sorted(glob.glob(annotated_glob, recursive=True))
    if not file_list:
        raise FileNotFoundError(f"No annotation files found matching pattern: {annotated_glob}")
        
    loaded_records = 0
    for fpath in file_list:
        with open(fpath, "r", encoding="utf-8") as fp:
            for line in fp:
                line_str = line.strip()
                if not line_str:
                    continue
                rec = json.loads(line_str)
                cid = rec.get("case_id", "").strip()
                target_d = rec.get("target_disease", "").strip()
                norm_d = normalize_disease_key(target_d)
                
                annotated_map[(cid, norm_d)] = rec
                annotated_by_case.setdefault(cid, []).append(rec)
                loaded_records += 1
                
    print(f"Indexed {loaded_records:,} annotated pairs across {len(file_list)} unique files.")
    print(f"Total unique (case_id, disease) pairs in map: {len(annotated_map):,}")
    print(f"Total distinct patient cases indexed: {len(annotated_by_case):,}")
    return annotated_map, annotated_by_case


# Load global ground truth database
annotated_map, annotated_by_case = load_annotated_database(ANNOTATED_DATA_GLOB)

Indexed 24,750 annotated pairs across 87 unique files.
Total unique (case_id, disease) pairs in map: 24,750
Total distinct patient cases indexed: 16,666


## 2. Tripartite Partitioning Engine (`partition_records`)

The function `partition_records(records, annotated_map, annotated_by_case=None)` classifies mismatch records into three mutually exclusive clinical cohorts:

1. **Group 2: Out of Scope Integrity (`OUT_OF_SCOPE`)**
   - Condition: MedCase `final_diagnosis` explicitly contains `"OUT_OF_SCOPE"` (case-insensitive).
   - Clinical Reality: The primary diagnosis lies outside the 21 target diseases. MedCase correctly refused to force-fit the patient into a closed taxonomy.
2. **Group 1: Multi-Pathogen Agreement (True Consensus Positives)**
   - Condition: MedCase `final_diagnosis` (or any extracted disease entity token from `parse_disease_entities`) matches a disease that 12-Step independently verified as `step12_dataset_label == "POSITIVE"` for the same patient.
   - Clinical Reality: Both systems agree on the verified affirmative pathogen (either acute complication or underlying chronic condition in multimorbid patients).
3. **Group 3: Genuine Diagnostic Divergence (True Clinical Dilemmas)**
   - Condition: All remaining mismatch cases.
   - Clinical Reality: True diagnostic dilemmas requiring judicial LLM arbitration (Claude / DeepSeek).

In [3]:
def partition_records(
    records: List[Dict[str, Any]], 
    annotated_map: Dict[Tuple[str, str], Dict[str, Any]], 
    annotated_by_case: Optional[Dict[str, List[Dict[str, Any]]]] = None
) -> Tuple[List[Any], List[Any], List[Any]]:
    """Partitions mismatch candidate records into three mutually exclusive clinical cohorts:
    - Group 1: Multi-Pathogen Agreement (MedCase final_diagnosis confirmed POSITIVE in 12-step)
    - Group 2: Out of Scope Integrity (final_diagnosis contains OUT_OF_SCOPE)
    - Group 3: Genuine Diagnostic Divergence (remaining clinical dilemma cases)
    
    Returns:
        group1: List of tuples (record_medcase, record_positive_12step, all_case_annotations)
        group2: List of tuples (record_medcase, all_case_annotations)
        group3: List of tuples (record_medcase, all_case_annotations)
    """
    if annotated_by_case is None:
        annotated_by_case = {}
        for (cid, _), rec in annotated_map.items():
            annotated_by_case.setdefault(cid, []).append(rec)
            
    group1: List[Tuple[Dict[str, Any], Dict[str, Any], List[Dict[str, Any]]]] = []
    group2: List[Tuple[Dict[str, Any], List[Dict[str, Any]]]] = []
    group3: List[Tuple[Dict[str, Any], List[Dict[str, Any]]]] = []
    
    for r in records:
        cid_full = r.get("case_id", "").strip()
        cid_base = cid_full.split("__")[0].strip()
        fd = r.get("final_diagnosis", "").strip()
        all_anns = annotated_by_case.get(cid_base, [])
        
        # Rule 2: Out of Scope Check
        if "out_of_scope" in fd.lower():
            group2.append((r, all_anns))
            continue
            
        # Rule 1: Multi-Pathogen Agreement Check (Tokenized Entity Set Matching)
        pred_entities = parse_disease_entities(fd) | {normalize_disease_key(fd)}
        match_pos_rec: Optional[Dict[str, Any]] = None
        
        for ann in all_anns:
            ann_d_norm = normalize_disease_key(ann.get("target_disease", ""))
            if ann.get("step12_dataset_label") == "POSITIVE":
                if any(ann_d_norm == pe or (len(pe) >= 4 and (ann_d_norm in pe or pe in ann_d_norm)) for pe in pred_entities if pe):
                    match_pos_rec = ann
                    break
                
        if match_pos_rec is not None:
            group1.append((r, match_pos_rec, all_anns))
        else:
            group3.append((r, all_anns))
            
    return group1, group2, group3

## 3. Clinical Forensic Helpers & Cognitive Error Framework (DEER Taxonomy)

To provide deep clinical interpretability across reports and judicial prompts without introducing bias, we implement:
- `format_evidence_block(r_12)`: Parses and extracts formatted quotes, strengths, and laboratory assays from `step9_key_evidence`.
- **Cognitive Error Classification (DEER Framework - Schiff et al. 2009, Graber et al. 2005):**
  * **Archetype A: DEER Class 3.2 (Faulty Synthesis, Premature Closure & Anchoring):** Anchoring on syndromic presentation while disregarding negative confirmatory tests or confusing historical cured disease (F4) with acute presentation (Sackett et al. 2000, Schiff et al. 2009, Uzuner et al. 2011).
  * **Archetype B: DEER Class 2.4 (Contextual Misattribution & Presumptive Closure):** Misattributing prophylactic vaccination (e.g. BCG scar), screening history (F5), or empiric treatment without laboratory proof (F6) as active disease (Harkema et al. 2009, Schiff et al. 2009).
  * **Archetype C: DEER Class 4.1 (Syndromic Look-Alike & Complex Differential Ambiguity):** True clinical mimicry or co-infection deconvolution without adequate microbiological resolution (Singhal et al. 2023, Tu et al. 2025).
- **Zero-Shot, Pure Double-Blind Adjudication Protocol:** To eliminate confirmation bias and leading the judge, cases are presented neutrally with zero hardcoded assumptions and no pre-classification hints (`<forensic_focus>` and hardcoded `archetype` tags eliminated). LLM arbiters independently derive the cognitive breakdown, classify archetypes, and detect failure traps inside `<clinical_scratchpad>` from first principles.

In [4]:
def format_evidence_block(r_12: Optional[Dict[str, Any]]) -> str:
    """Extracts and formats laboratory evidence items from a 12-step annotation record."""
    if not r_12:
        return "- None documented (Negative / Ruled Out)"
    raw_ev = r_12.get("step9_key_evidence", "")
    if isinstance(raw_ev, dict):
        ev_items = raw_ev.get("evidence_items", [])
        if not ev_items:
            return "- None documented (Negative / Ruled Out)"
        formatted = []
        for it in ev_items:
            quote = it.get("source_quote", "").strip()
            test = it.get("test_name", "Diagnostic Finding")
            st = it.get("strength", "EVIDENCE")
            res = it.get("result", "")
            formatted.append(f"- **[{st}] {test}** ({res}): \"{quote}\"")
        return "\n".join(formatted) if formatted else "- None documented"
    elif isinstance(raw_ev, str) and raw_ev.strip():
        return f"- \"{raw_ev.strip()}\""
    return "- None documented"

## 4. Clinical Cross-Validation Report Generators

This section formats and exports:
1. `group1_cross_disease_agreement.md`: Complete dossier of cases where both systems identified validated diagnoses.
2. `group2_out_of_scope.md`: Complete dossier of out-of-scope refusals where closed-taxonomy failure was properly avoided.
3. `group3_diagnostic_divergence.md`: Complete dossier of genuine clinical diagnostic dilemmas.
4. `tripartite_summary.json`: High-level cohort metrics and case lists.

In [5]:
def build_report_group1(group1: List[Any], total_count: int, split_name: str) -> str:
    """Builds the Markdown report for Group 1: Multi-Pathogen Agreement."""
    n_g1 = len(group1)
    pct = (n_g1 / total_count * 100) if total_count else 0
    lines = []
    lines.append(f"# Group 1 Clinical Cross-Validation Report: Multi-Pathogen Agreement ({n_g1} Cases)")
    lines.append(f"> **Cohort:** `{split_name}` | **Total Split Cases:** {total_count} | **Agreement Count:** {n_g1} ({pct:.1f}%)")
    lines.append(">")
    lines.append("> **Clinical Reality:** Both models agree on the definitive underlying/acute pathogen. Discrepancies were artifacts of closed multiclass testing vs open hypothesis validation.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 1. Executive Summary & Cross-Disease Agreement Matrix")
    lines.append("")
    lines.append("| # | Case ID | Original Evaluated Disease | 12-Step Original Label | MedCase Predicted Disease | 12-Step Label for Predicted Disease | Clinical Evidence Strength | Agreement Mechanism |")
    lines.append("|:---:|:---|:---:|:---:|:---:|:---:|:---:|:---|")
    
    for idx, (r_mc, r_pos_12, all_anns) in enumerate(group1, 1):
        cid = r_mc.get("case_id", "")
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        pred_l = r_pos_12.get("step12_dataset_label", "POSITIVE")
        ev_str = r_pos_12.get("step9_evidence_strength", "E1_CONFIRMATORY")
        mech = "Opportunistic Coinfection" if orig_l == "POSITIVE" else "Look-Alike Ruled-Out with True Positive Identified"
        lines.append(f"| {idx} | `{cid}` | {orig_d} | **{orig_l}** | **`{pred_d}`** | **`{pred_l}`** | `{ev_str}` | {mech} |")
        
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 2. Case Dossiers")
    lines.append("")
    
    for idx, (r_mc, r_pos_12, all_anns) in enumerate(group1, 1):
        cid = r_mc.get("case_id", "")
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        prompt = r_mc.get("case_prompt", "").strip()
        analysis = r_mc.get("case_analysis", "").strip()
        reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()
        
        pred_role = r_pos_12.get("step8_clinical_role", "PRIMARY_DIAGNOSIS")
        pred_ev_str = r_pos_12.get("step9_evidence_strength", "E1_CONFIRMATORY")
        pred_rationale = r_pos_12.get("rationale", "").strip()
        pred_ev_block = format_evidence_block(r_pos_12)
        
        lines.append(f"### Case {idx}: `{cid}`")
        lines.append(f"- **Evaluated Hypothesis:** `{orig_d}` ({orig_l})")
        lines.append(f"- **MedCase Selection:** **`{pred_d}`**")
        lines.append(f"- **12-Step Cross-Verification for `{pred_d}`:** **CONFIRMED POSITIVE** (Role: `{pred_role}`, Strength: `{pred_ev_str}`)")
        lines.append("")
        lines.append("#### Multi-Condition Landscape for Patient:")
        for a in all_anns:
            lines.append(f"- `{a.get('target_disease')}`: **{a.get('step12_dataset_label')}** (Role: `{a.get('step8_clinical_role')}`)")
        lines.append("")
        lines.append("#### Pre-Diagnostic Clinical Presentation:")
        lines.append("> " + prompt.replace("\n", "\n> "))
        lines.append("")
        lines.append("#### MedCase Reasoning Trajectory:")
        lines.append("```text")
        lines.append(analysis)
        lines.append("```")
        lines.append(reasoning_mc)
        lines.append("")
        lines.append(f"#### MultiCaRe 12-Step Verification on `{pred_d}`:")
        lines.append(f"- **Rationale:** {pred_rationale}")
        lines.append(f"- **Evidence:**\n{pred_ev_block}")
        lines.append("")
        lines.append("---")
        lines.append("")
        
    return "\n".join(lines)


def build_report_group2(group2: List[Any], total_count: int, split_name: str) -> str:
    """Builds the Markdown report for Group 2: Out of Scope Integrity."""
    n_g2 = len(group2)
    pct = (n_g2 / total_count * 100) if total_count else 0
    lines = []
    lines.append(f"# Group 2 Clinical Cross-Validation Report: Out-of-Scope Integrity ({n_g2} Cases)")
    lines.append(f"> **Cohort:** `{split_name}` | **Total Split Cases:** {total_count} | **Out-of-Scope Refusal Count:** {n_g2} ({pct:.1f}%)")
    lines.append(">")
    lines.append("> **Clinical Reality:** Patients presented with rare, malignant, or unlisted conditions (e.g. Kaposi sarcoma, sporotrichosis, DILI). MedCase accurately refused closed-domain misclassification, while 12-Step accurately audited specific candidate hypotheses.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 1. Executive Summary & Out-of-Scope Condition Taxonomy")
    lines.append("")
    lines.append("| # | Case ID | Original Target Hypothesis | 12-Step Clinical Label | MedCase Out-of-Scope Identification | Primary Clinical Pathology |")
    lines.append("|:---:|:---|:---:|:---:|:---|:---|")
    
    for idx, (r_mc, all_anns) in enumerate(group2, 1):
        cid = r_mc.get("case_id", "")
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        pathology = pred_d.replace("-> OUT_OF_SCOPE", "").replace("-> out_of_scope", "").strip()
        lines.append(f"| {idx} | `{cid}` | {orig_d} | **{orig_l}** | `{pred_d}` | **{pathology}** |")
        
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 2. Case Dossiers")
    lines.append("")
    
    for idx, (r_mc, all_anns) in enumerate(group2, 1):
        cid = r_mc.get("case_id", "")
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        prompt = r_mc.get("case_prompt", "").strip()
        analysis = r_mc.get("case_analysis", "").strip()
        reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()
        
        lines.append(f"### Case {idx}: `{cid}`")
        lines.append(f"- **Evaluated Candidate Hypothesis:** `{orig_d}` ({orig_l})")
        lines.append(f"- **MedCase Refusal / Diagnosis:** **`{pred_d}`**")
        lines.append("")
        lines.append("#### Multi-Condition Landscape for Patient:")
        for a in all_anns:
            lines.append(f"- `{a.get('target_disease')}`: **{a.get('step12_dataset_label')}** ({a.get('step8_clinical_role')})")
        lines.append("")
        lines.append("#### Pre-Diagnostic Presentation:")
        lines.append("> " + prompt.replace("\n", "\n> "))
        lines.append("")
        lines.append("#### MedCase Analysis:")
        lines.append("```text")
        lines.append(analysis)
        lines.append("```")
        lines.append(reasoning_mc)
        lines.append("")
        lines.append("---")
        lines.append("")
        
    return "\n".join(lines)


def build_report_group3(group3: List[Any], total_count: int, split_name: str, pair_index: Dict[Tuple[str, str], Dict[str, Any]]) -> str:
    """Builds the Markdown report for Group 3: Genuine Diagnostic Divergence with DEER Cognitive Error Grounding."""
    n_g3 = len(group3)
    pct = (n_g3 / total_count * 100) if total_count else 0
    lines = []
    lines.append(f"# Group 3 Clinical Cross-Validation Report: Diagnostic Divergence & Conundrums ({n_g3} Cases)")
    lines.append(f"> **Cohort:** `{split_name}` | **Total Split Cases:** {total_count} | **True Divergence Count:** {n_g3} ({pct:.1f}%)")
    lines.append(">")
    lines.append("> **Clinical Reality:** Hard differential diagnostic dilemmas where one system favored a pathogen that objective testing excluded, or where viral/parasitic mimics created diagnostic ambiguity. Primary benchmark for LLM judicial arbitration.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 1. Cognitive Diagnostic Breakdown Framework (DEER Taxonomy)")
    lines.append("")
    lines.append("Cases in Group 3 represent true clinical conundrums evaluated under the Diagnostic Error Evaluation and Research (DEER) taxonomy (Schiff et al. 2009, Graber et al. 2005):")
    lines.append("- **Archetype A: DEER Class 3.2 (Faulty Synthesis, Premature Closure & Anchoring):** Grounded in Sackett et al. (2000), Schiff et al. (2009), Uzuner et al. (2011). One system anchors on syndromic presentation while overriding definitive negative laboratory tests or confusing remote cured disease (F4) with acute etiology.")
    lines.append("- **Archetype B: DEER Class 2.4 (Contextual Misattribution & Presumptive Closure):** Grounded in Harkema et al. (2009), Schiff et al. (2009). Misattributing prophylactic vaccination (F5), screening, or empiric treatment without laboratory confirmation (F6) as active disease.")
    lines.append("- **Archetype C: DEER Class 4.1 (Syndromic Look-Alike & Complex Differential Ambiguity):** Grounded in Singhal et al. (2023), Tu et al. (2025). Severe overlapping syndromic presentations or viral bystander reactivation in sepsis.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 2. Executive Summary & Divergent Cases")
    lines.append("")
    lines.append("| # | Case ID | Original Disease Hypothesis | 12-Step Clinical Label | MedCase Prediction | 12-Step Status for Predicted Entity |")
    lines.append("|:---:|:---|:---:|:---:|:---:|:---:|")
    
    for idx, (r_mc, all_anns) in enumerate(group3, 1):
        cid = r_mc.get("case_id", "")
        cid_base = cid.split("__")[0].strip()
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        
        r_12_pred = pair_index.get((cid_base, normalize_disease_key(pred_d)))
        if not r_12_pred:
            for pe in parse_disease_entities(pred_d):
                if (cid_base, pe) in pair_index:
                    r_12_pred = pair_index[(cid_base, pe)]
                    break
        
        status_for_pred = "Not Formally Annotated"
        if r_12_pred:
            status_for_pred = f"{r_12_pred.get('step12_dataset_label')} ({r_12_pred.get('step8_clinical_role')})"
            
        lines.append(f"| {idx} | `{cid}` | {orig_d} | **{orig_l}** | **`{pred_d}`** | `{status_for_pred}` |")
        
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## 3. Forensic Case Dossiers")
    lines.append("")
    
    for idx, (r_mc, all_anns) in enumerate(group3, 1):
        cid = r_mc.get("case_id", "")
        cid_base = cid.split("__")[0].strip()
        orig_d = r_mc.get("target_disease", "")
        orig_l = r_mc.get("clinical_label", "")
        pred_d = r_mc.get("final_diagnosis", "")
        prompt = r_mc.get("case_prompt", "").strip()
        analysis = r_mc.get("case_analysis", "").strip()
        reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()
        
        r_12_hyp = pair_index.get((cid_base, normalize_disease_key(orig_d)))
        
        orig_role = r_12_hyp.get("step8_clinical_role", "N/A") if r_12_hyp else "N/A"
        orig_ev = r_12_hyp.get("step9_evidence_strength", "N/A") if r_12_hyp else "N/A"
        orig_rationale = r_12_hyp.get("rationale", "N/A") if r_12_hyp else "N/A"
        orig_ev_block = format_evidence_block(r_12_hyp)
        
        lines.append(f"### Case {idx}: `{cid}`")
        lines.append(f"- **Evaluated Hypothesis (MultiCaRe):** `{orig_d}` (12-Step: **{orig_l}**, Role: `{orig_role}`, Evidence: `{orig_ev}`)")
        lines.append(f"- **MedCase Primary Selection:** **`{pred_d}`**")
        lines.append(f"- **Diagnostic Tension Profiling:** Divergence between targeted hypothesis `{orig_d}` and multiclass choice `{pred_d}`.")
        lines.append(f"- **Cognitive Error Context (DEER):** Evaluated against DEER Class 3.2 (Synthesis/Anchoring), Class 2.4 (Contextual/Testing), Class 4.1 (Premature Closure).")
        lines.append(f"- **Literature Grounding:** SpPIn/SnNOut heuristics (Sackett 2000), DEER taxonomy (Schiff 2009), Assertion temporality (Uzuner 2011; Harkema 2009).")
        lines.append("")
        lines.append("#### Multi-Condition Landscape for Patient:")
        for a in all_anns:
            lines.append(f"- `{a.get('target_disease')}`: **{a.get('step12_dataset_label')}** (Role: `{a.get('step8_clinical_role')}`, Evidence: `{a.get('step9_evidence_strength')}`)")
        lines.append("")
        lines.append("#### Clinical Presentation:")
        lines.append("> " + prompt.replace("\n", "\n> "))
        lines.append("")
        lines.append("#### MedCase Trajectory:")
        lines.append("```text")
        lines.append(analysis)
        lines.append("```")
        lines.append(reasoning_mc)
        lines.append("")
        lines.append(f"#### MultiCaRe 12-Step Grounding on `{orig_d}`:")
        lines.append(f"- **Rationale:** {orig_rationale}")
        lines.append(f"- **Evidence Items:**\n{orig_ev_block}")
        lines.append("")
        lines.append("---")
        lines.append("")
        
    return "\n".join(lines)


def build_tripartite_summary_json(total_count: int, group1: List[Any], group2: List[Any], group3: List[Any], split_name: str) -> Dict[str, Any]:
    """Compiles split-level tripartite metrics into a clean structured dictionary."""
    g1_count = len(group1)
    g2_count = len(group2)
    g3_count = len(group3)
    return {
        "split_name": split_name,
        "total_cases": total_count,
        "group1_multi_pathogen_agreement": {
            "count": g1_count,
            "percentage": round(g1_count / total_count * 100, 2) if total_count else 0.0,
            "description": "MedCase predicted disease was independently confirmed POSITIVE by 12-Step pipeline.",
            "case_ids": [r.get("case_id") for r, _, _ in group1]
        },
        "group2_out_of_scope_integrity": {
            "count": g2_count,
            "percentage": round(g2_count / total_count * 100, 2) if total_count else 0.0,
            "description": "MedCase correctly recognized that primary disease was outside the 21 target list.",
            "case_ids": [r.get("case_id") for r, _ in group2]
        },
        "group3_genuine_diagnostic_divergence": {
            "count": g3_count,
            "percentage": round(g3_count / total_count * 100, 2) if total_count else 0.0,
            "description": "Genuine clinical diagnostic conflicts evaluated by independent LLM judges.",
            "case_ids": [r.get("case_id") for r, _ in group3]
        }
    }

## 5. Anthropic Claude Forensic Adjudication Prompt Generator

This generator compiles copy-ready, production-grade prompt packages specifically structured for **Anthropic Claude (Claude Sonnet 5.5)** following Anthropic prompt-engineering best practices:
- **XML Tag Demarcation:** `<system_instructions>`, `<evaluation_task>`, `<case>`, `<case_metadata>`, `<clinical_case_presentation>`, `<method_x_reasoning>`, `<method_y_grounding>`, `<clinical_scratchpad>`, `<forensic_adjudication>`.
- **Senior Clinical Forensic Arbiter Persona:** Strict evidence-first stance.
- **Anti-Sycophancy Directives:** 4 hard prohibitions forbidding diplomatic hedging.
- **Epistemic Literature Grounding ($D_1 	o D_5$):** Directly cites Singhal 2023, Tu 2025, Sackett 2000, Schiff 2009, Uzuner 2011, Harkema 2009, Goyal 2025, Singh 2014.
- **Pure Double-Blind Adjudication:** Zero hardcoded assumptions or leading hints (`<forensic_focus>` and `archetype="` eliminated).
- **5-Step `<clinical_scratchpad>` Audit:** Enforces sequential verification before scoring.
- **Late-Binding JSON Output Schema:** Guarantees machine-parseable judicial decisions.

In [6]:
def build_claude_system_prompt() -> str:
    """Constructs Anthropic Claude System Prompt with clinical forensic persona and hard anti-sycophancy rules."""
    return """<system_instructions>

<persona>
You are a Senior Clinical Forensic Arbiter — an elite physician-scientist with dual board certification in
Infectious Disease and Clinical Pathology, and extensive expertise in Diagnostic Decision Systems and Evidence-Based Medicine.
You chair hospital Diagnostic Quality Assurance Boards and specialize in resolving high-stakes clinical diagnostic conflicts
between autonomous clinical reasoning algorithms. Your adjudications are rigorous, evidence-grounded, and defensible at grand rounds.
</persona>

<analytical_stance>
You are strictly EVIDENCE-FIRST, not narrative-first. Diagnostic intuition, syndromic plausibility, and travel history are
hypothesis generators, NEVER substitutes for objective microbiological verification.

Under evidence-based diagnostic heuristics (Sackett et al., Evidence-Based Medicine 2000; Schiff et al., Arch Intern Med 2009):
1. Definitive negative gold-standard laboratory testing (negative smears, negative cultures, non-reactive specific serology/PCR,
   or contradicting biopsy findings) conclusively rules OUT the disease under SpPIn/SnNOut principles. Affirming an excluded diagnosis
   constitutes an unacceptable breach of evidence-based clinical reasoning.
2. Inactive past medical history cured years/decades prior (F4) MUST NOT be affirmed as the etiology of an acute current illness
   (Uzuner et al., JAMIA 2011).
3. Prophylactic interventions, vaccination scars (e.g. BCG scar), and negative screening history (F5) MUST NOT be confused
   with active pathologic infection (Harkema et al., JBI 2009).
4. Presumptive or empiric clinical therapy without laboratory confirmation (F6) must be annotated as UNRESOLVED / PRESUMPTIVE,
   never as an established definitive gold-standard positive (CDC MMWR Case Definitions 1997).

You aggressively detect and penalize:
- FALSE POSITIVE RULE-IN errors (DEER Class 2.4/3.2): Affirming a diagnosis despite definitive negative lab testing.
- TEMPORALITY CONFUSION (DEER Class 3.2 / F4): Confusing cured historical disease with current acute presentation.
- DISCOURSE SCOPE DRIFT (DEER Class 2.4 / F5): Confusing vaccines/prophylaxis with active infection.
- DIPLOMATIC HEDGING: Unwillingness to declare an objective winner when one system is scientifically superior.
- VERBOSITY BIAS: Do not reward verbose reasoning traces that arrive at an ungrounded or contradicted conclusion.
</analytical_stance>

<epistemic_literature_grounding>
Your judicial evaluation must strictly embody peer-reviewed diagnostic error and clinical AI consensus:
- Dimension 1 (Diagnostic Accuracy, D1): Grounded in Singhal et al. (Med-PaLM, Nature 2023) and Tu et al. (AMIE, Nature 2025). Evaluates identification of true etiologic pathogen confirmed by CDC/WHO criteria [E1/E2].
- Dimension 2 (Rule-Out Rigor, D2): Grounded in Sackett et al. (EBM 2000; SpPIn/SnNOut) and Schiff et al. (DEER Taxonomy, Arch Intern Med 2009). Disregarding definitive negative laboratory or biopsy testing constitutes a catastrophic diagnostic error (score <= 2).
- Dimension 3 (Temporality & Context, D3): Grounded in Uzuner et al. (i2b2 Assertion, JAMIA 2011) and Harkema et al. (ConText, JBI 2009). Demands strict discrimination between acute pathology vs cured past history [F4] or prophylactic vaccination/screening [F5].
- Dimension 4 (Evidence Grounding, D4): Grounded in Goyal et al. / CliniFact (JAMIA 2025). Demands strict factual alignment between generated clinical claims and source case text, heavily penalizing hallucinated labs or findings.
- Dimension 5 (Patient Safety, D5): Grounded in Singh & Graber (Diagnosis 2014) and Med-PaLM Harm Stratification Matrix (Nature 2023). Penalizes preventable patient harm, toxic unneeded therapy cascades, and missed life-threatening infections.
</epistemic_literature_grounding>

<evidence_grading_framework>
You apply the MultiCaRe Evidence Hierarchy grounded in CDC MMWR Case Definitions (1997) and WHO surveillance criteria in all assessments:
- E1_CONFIRMATORY_LAB: Definitive gold-standard pathogen proof (culture, PCR, biopsy histopathology with organism, direct visualization).
- E2_HIGHLY_SPECIFIC: Highly specific validated assays (specific antigen EIA, IgM seroconversion, IgG avidity).
- E3_SUGGESTIVE_NON_SPECIFIC: Clinical syndrome, non-specific lab trends, imaging patterns — suggestive but INSUFFICIENT alone for definitive confirmation.
- E0_CONTRADICTED: Objective testing performed and explicitly NEGATIVE — disease is RULED OUT unless proven false-negative.
- E0_NO_EVIDENCE: Diagnostic testing absent or non-confirmatory — absence of evidence is not evidence of disease.
- E4_CONTEXT: Mentions strictly relating to background, differential discussion, cured history, or vaccination (F4, F5).
</evidence_grading_framework>

<anti_sycophancy_directives>
The following are HARD PROHIBITIONS — violating any of these constitutes an adjudication failure:
- PROHIBITION 1: NEVER split the difference or declare a diplomatic tie when one method is objectively superior on evidence (Sackett et al. 2000).
- PROHIBITION 2: NEVER accept narrative plausibility over negative laboratory results. If a test is explicitly negative, any method claiming that disease is POSITIVE has committed an egregious diagnostic failure (Schiff et al. 2009).
- PROHIBITION 3: NEVER penalize Method Y for correctly annotating a disease as NEGATIVE simply because the patient presented with symptoms resembling that disease.
- PROHIBITION 4: NEVER confuse historical conditions (e.g., childhood TB cured 20 years ago) with active presenting diagnoses (Uzuner et al. 2011; Harkema et al. 2009).
</anti_sycophancy_directives>

<scoring_rubric>
Score each method independently on 5 dimensions (1 to 5 integer scale):

D1: diagnostic_accuracy (1-5)
  5: Accurately identifies true etiologic condition backed by definitive gold-standard evidence (E1/E2).
  3: Plausible differential but misses primary etiologic pathogen or over-specifies without proof.
  1: Entirely incorrect, hallucinated diagnosis, or contradicts confirmed laboratory reality.

D2: rule_out_rigor (1-5)
  5: Flawlessly respects negative laboratory/biopsy findings; correctly excludes non-viable candidates.
  3: Incomplete exclusion; leaves ruled-out conditions ambiguous.
  1: Catastrophic false-positive rule-in; affirms disease directly contradicted by negative lab/pathology.

D3: temporality_context (1-5)
  5: Perfectly differentiates active acute illness from remote history (F4) and prophylactic exposure (F5).
  3: Minor ambiguity in temporal anchoring but reaches acceptable conclusion.
  1: Gross conflation of remote past history or vaccination with acute active pathology.

D4: evidence_grounding (1-5)
  5: Every assertion is directly anchored in documented case facts; zero hallucination.
  3: Mostly grounded but includes unverified clinical extrapolations.
  1: Severe hallucination of clinical facts, tests, or results not in the source text.

D5: patient_safety (1-5)
  5: Adjudication leads to correct, targeted management; minimizes unnecessary toxicity.
  3: Suboptimal management path with moderate risk of delayed treatment or overtreatment.
  1: Dangerous failure; triggers toxic unneeded therapies (e.g. toxic leprosy MDT) or misses lethal disease.
</scoring_rubric>

<mandatory_workflow>
For each case, you MUST execute your analysis sequentially inside `<clinical_scratchpad>` before producing the final JSON verdict:
Step 1: Clinical Fact Extraction — Extract all objective laboratory tests, imaging, and biopsy findings with exact results.
Step 2: Method X (MedCase) Audit — Evaluate whether Method X's diagnosis is confirmed, contradicted, or unproven. Check for F1-F6 errors.
Step 3: Method Y (12-Step) Audit — Evaluate whether Method Y's labels are supported by the MultiCaRe evidence hierarchy. Check for F1-F6 errors.
Step 4: Tension Archetype Classification — Determine whether this case exemplifies Archetype A (Rule-in vs Rule-out/Temporality), Archetype B (Context/Prophylaxis/Presumptive), or Archetype C (Syndromic mimic/Bystander).
Step 5: Forensic Synthesis — Determine the definitively superior method with zero diplomatic hedging.
</mandatory_workflow>

<output_format>
Your final output for each case MUST be valid JSON enclosed in `<forensic_adjudication>` tags matching this schema:
```json
{
  "case_id": "string",
  "dilemma_archetype": "ARCHETYPE_A | ARCHETYPE_B | ARCHETYPE_C",
  "method_x_audit": {
    "predicted_diagnosis": "string",
    "clinical_status": "CONFIRMED_CORRECT | FALSE_POSITIVE_RULE_IN | HISTORICAL_CONFLATION | PROPHYLAXIS_CONFUSION | PLAUSIBLE_BUT_UNPROVEN | INCORRECT",
    "failure_modes": ["F1_LEXICAL_HOMONYM | F2_DOCUMENT_SCOPE | F3_PATIENT_ANCHORING | F4_TEMPORALITY_BIAS | F5_DISCOURSE_CONTEXT | F6_SPECIFICITY_GAP"],
    "scores": {"D1": int, "D2": int, "D3": int, "D4": int, "D5": int, "total": int}
  },
  "method_y_audit": {
    "evaluated_hypotheses": [{"disease": "string", "label": "POSITIVE | NEGATIVE", "verdict_soundness": "CORRECT | ERRONEOUS"}],
    "failure_modes": [],
    "scores": {"D1": int, "D2": int, "D3": int, "D4": int, "D5": int, "total": int}
  },
  "judicial_verdict": {
    "winner": "METHOD_X | METHOD_Y | BOTH_PARTIALLY_CORRECT | NEITHER_CORRECT",
    "definitive_explanation": "string (evidence-based justification citing specific lab findings)",
    "prevented_clinical_harm": "string (what clinical harm was avoided by the winning method)"
  }
}
```
</output_format>

Think the problem through before you answer.
</system_instructions>"""


def build_claude_user_header(n_cases: int) -> str:
    """Constructs the user prompt header for Claude batch evaluation."""
    return f"""<evaluation_task>
<task_overview>
You are adjudicating {n_cases} clinical cases exhibiting genuine diagnostic divergence between:
- Method X: Stanford MedCaseReasoning (GPT-OSS 120B) — Multiclass diagnostic generation model.
- Method Y: MultiCaRe 12-Step Clinical Annotation Pipeline — Targeted hypothesis verification engine.

For each case:
1. Conduct an unconstrained clinical forensic analysis inside `<clinical_scratchpad>`.
2. Apply the 5-dimensional scoring rubric (D1 to D5, 1-5 integer scale).
3. Deliver a definitive, unambiguous judicial verdict in the specified JSON format inside `<forensic_adjudication>`.
No diplomatic ties. Follow the anti-sycophancy directives strictly.
</task_overview>"""


def format_claude_case_entry(
    idx: int, 
    r_mc: Dict[str, Any], 
    pair_index: Dict[Tuple[str, str], Dict[str, Any]], 
    all_by_case: Dict[str, List[Dict[str, Any]]]
) -> str:
    """Formats a single case into Anthropic XML case tags for Claude evaluation.
    Strictly Double-Blind: No hardcoded case overrides, no leading archetype, no leading forensic hints.
    """
    cid = r_mc.get("case_id", "")
    cid_base = cid.split("__")[0].strip()
    td = r_mc.get("target_disease", "")
    cl = r_mc.get("clinical_label", "")
    fd = r_mc.get("final_diagnosis", "")
    prompt = r_mc.get("case_prompt", "").strip()
    analysis = r_mc.get("case_analysis", "").strip()
    reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()
    
    all_anns = all_by_case.get(cid_base, [])
    r_12_hyp = pair_index.get((cid_base, normalize_disease_key(td)))
    
    # Resolve predicted entity using set tokenization if direct lookup returns None
    r_12_pred = pair_index.get((cid_base, normalize_disease_key(fd)))
    if not r_12_pred:
        for pe in parse_disease_entities(fd):
            if (cid_base, pe) in pair_index:
                r_12_pred = pair_index[(cid_base, pe)]
                break
    
    label_hyp = r_12_hyp.get("step12_dataset_label", cl) if r_12_hyp else cl
    role_hyp = r_12_hyp.get("step8_clinical_role", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    ev_str_hyp = r_12_hyp.get("step9_evidence_strength", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    rationale_hyp = r_12_hyp.get("rationale", "") if r_12_hyp else ""
    
    label_pred = r_12_pred.get("step12_dataset_label", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    role_pred = r_12_pred.get("step8_clinical_role", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    ev_str_pred = r_12_pred.get("step9_evidence_strength", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    rationale_pred = r_12_pred.get("rationale", "") if r_12_pred else ""
    
    all_hyps_block = ""
    if all_anns:
        hyp_lines = []
        for ann in all_anns:
            d_name = ann.get("target_disease", "UNKNOWN")
            d_lbl = ann.get("step12_dataset_label", "UNKNOWN")
            d_role = ann.get("step8_clinical_role", "UNKNOWN")
            d_str = ann.get("step9_evidence_strength", "UNKNOWN")
            hyp_lines.append(f"  * {d_name}: {d_lbl} (Role: {d_role}, Strength: {d_str})")
        all_hyps_block = "\n".join(hyp_lines)
    else:
        all_hyps_block = f"  * {td}: {cl}"

    lines = []
    # Pure Double-Blind: Zero leading archetype or forensic focus attributes
    lines.append(f'<case id="{cid}" index="{idx}">')
    lines.append("<case_metadata>")
    lines.append(f"- **Patient Case ID:** `{cid_base}`")
    lines.append(f"- **Target Hypothesis Audited by Pipeline:** `{td}`")
    lines.append(f"- **Ground Truth Annotation Label:** `{cl}`")
    lines.append(f"- **Method X (MedCase GPT-OSS 120B) Selected Diagnosis:** **`{fd}`**")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Verdict for `{td}`:** `{label_hyp}` (Role: `{role_hyp}`, Strength: `{ev_str_hyp}`)")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Verdict for `{fd}`:** `{label_pred}` (Role: `{role_pred}`, Strength: `{ev_str_pred}`)")
    lines.append("</case_metadata>")
    lines.append("")
    lines.append("<all_12step_hypotheses_evaluated>")
    lines.append(all_hyps_block)
    lines.append("</all_12step_hypotheses_evaluated>")
    lines.append("")
    lines.append("<clinical_case_presentation>")
    lines.append(prompt)
    lines.append("</clinical_case_presentation>")
    lines.append("")
    lines.append("<method_x_reasoning>")
    lines.append(f"Primary Diagnosis Selected: {fd}")
    lines.append("Breakpoint & Scope Analysis:")
    lines.append(analysis)
    lines.append("Differential Diagnostic Reasoning:")
    lines.append(reasoning_mc)
    lines.append("</method_x_reasoning>")
    lines.append("")
    lines.append("<method_y_grounding>")
    lines.append(f"- Adjudication for `{td}`: {label_hyp} (Role: {role_hyp}, Evidence: {ev_str_hyp})")
    lines.append(f"- Adjudication for `{fd}`: {label_pred} (Role: {role_pred}, Evidence: {ev_str_pred})")
    lines.append("- Direct Evidence Extracted for Hypothesis:")
    lines.append(format_evidence_block(r_12_hyp))
    if r_12_pred and r_12_pred != r_12_hyp:
        lines.append(f"- Direct Evidence Extracted for Predicted Entity (`{fd}`):")
        lines.append(format_evidence_block(r_12_pred))
    lines.append("- Clinical Rationale:")
    lines.append(rationale_hyp or rationale_pred or "Evidence evaluated against 12-step protocol.")
    lines.append("</method_y_grounding>")
    lines.append("")
    lines.append("</case>")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


def generate_claude_prompt(
    group3: List[Any], 
    pair_index: Dict[Tuple[str, str], Dict[str, Any]], 
    all_by_case: Dict[str, List[Dict[str, Any]]], 
    split_name: str
) -> str:
    """Generates the complete Claude evaluation markdown benchmark file."""
    n_cases = len(group3)
    lines = []
    lines.append(f"# Claude Forensic Evaluation Benchmark: Group 3 Diagnostic Divergence ({split_name})")
    lines.append("")
    lines.append("> **Anthropic Claude Best-Practice Evaluation Benchmark**")
    lines.append(f"> Designed for forensic adjudication of the **{n_cases} genuine clinical diagnostic divergence cases** in split `{split_name}`.")
    lines.append("> - **Method X:** Stanford MedCaseReasoning (GPT-OSS 120B multiclass model)")
    lines.append("> - **Method Y:** MultiCaRe 12-Step Clinical Annotation (Evidentiary verification protocol)")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("## 1. SYSTEM PROMPT (Copy into Claude 'System' field)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")
    lines.append("```xml")
    lines.append(build_claude_system_prompt())
    lines.append("```")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("## 2. USER PROMPT & CASE DOSSIERS (Copy into 'User' field)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")
    lines.append("```xml")
    lines.append(build_claude_user_header(n_cases))
    lines.append("")
    lines.append("<!-- CASE DATA BLOCKS -->")
    lines.append("")
    for idx, (r_mc, _) in enumerate(group3, 1):
        lines.append(format_claude_case_entry(idx, r_mc, pair_index, all_by_case))
    lines.append("```")
    lines.append("")
    return "\n".join(lines)

## 6. DeepSeek (R1 / DeepThink) Forensic Judicial Prompt Generator

This generator compiles copy-ready judicial prompts optimized for **DeepSeek-R1 (DeepThink)** and **DeepSeek-V3**:
- **Chief Medical Officer Persona:** Unconstrained clinical reasoning trace (`<think>`) to audit complex causal conflicts and diagnostic contradictions.
- **Epistemic Literature & DEER Grounding:** Evaluates Diagnostic Accuracy ($D_1$, Singhal 2023), Rule-Out Rigor ($D_2$, Sackett 2000), Temporality ($D_3$, Uzuner 2011), Grounding ($D_4$, Goyal 2025), and Safety ($D_5$, Singh 2014) alongside DEER Classes 3.2, 2.4, 4.1.
- **Pure Double-Blind Adjudication:** Zero hardcoded assumptions or leading hints.
- **Comparative Scorecard Table:** Markdown comparison matrix per case.
- **Definitive Verdict Protocol:** Requires explicit declaration of Winner and medical justification.

In [7]:
def build_deepseek_header(n_cases: int, split_name: str) -> str:
    """Constructs DeepSeek judicial header with CMO persona and evaluation rubric."""
    return f"""# DeepSeek Clinical Adjudication Benchmark: {n_cases} Genuine Diagnostic Divergence Cases ({split_name})

> **INSTRUCTION FOR DEEPSEEK (DEEPTHINK / R1):**
> You are an expert Chief Medical Officer, Infectious Disease Specialist, and Master Forensic Arbiter.
> Below are **{n_cases} REAL CLINICAL CASES** representing genuine diagnostic divergence between:
> - **Method X:** Stanford MedCaseReasoning (GPT-OSS 120B) — Multiclass diagnostic generation model.
> - **Method Y:** MultiCaRe 12-Step Clinical Annotation — Rigorous targeted evidentiary verification engine.
>
> In all cases, the two systems reached opposing conclusions regarding disease presence, rule-out status, or diagnostic attribution.
> Conduct an impartial, rigorous, clinical and microbiological forensic trial to determine which system is medically correct.

---

## Forensic Evaluation Rubric (Rate 1 to 5 for each method)

1. **Diagnostic Accuracy & Consensus Alignment ($D_1$, 1–5):** (Singhal et al., Nature 2023; Tu et al., Nature 2025) Does the method identify the true etiologic pathogen supported by CDC/WHO confirmatory laboratory standards (E1/E2)?
2. **Rule-Out Rigor vs False-Positive Rule-In ($D_2$, 1–5):** (Sackett et al., 2000; Schiff et al., DEER Class 2.4, 2009) Does the method properly respect definitive negative laboratory tests (e.g. negative blood smears, negative biopsies), or does it commit confirmation bias?
3. **Temporality & Discourse Context Parsing ($D_3$, 1–5):** (Uzuner et al., JAMIA 2011; Harkema et al., 2009; DEER Class 3.2) Does the method distinguish between active current illness vs resolved past medical history (F4) or prophylactic vaccination (F5)?
4. **Evidence Grounding & Verbatim Fidelity ($D_4$, 1–5):** (Goyal et al. / CliniFact, JAMIA 2025) Are claims strictly anchored in objective clinical findings from the text without hallucinated clinical assumptions?
5. **Patient Safety & Diagnostic Prudence ($D_5$, 1–5):** (Singh & Graber, Diagnosis 2014) Would acting on this method's diagnosis protect the patient, or lead to toxic/unnecessary therapies (e.g. toxic anti-leprosy multidrug therapy, toxic anti-helminthics)?

---

## Diagnostic Dilemma Archetypes & Pitfalls (DEER Cognitive Taxonomy)
- **Archetype A (Rule-In vs Rule-Out / Temporality - DEER Class 3.2):** Inactive remote past history (F4) or affirmation despite negative confirmatory testing (F6).
- **Archetype B (Context / Prophylaxis / Presumptive - DEER Class 2.4 & 3.2):** Prophylactic vaccination or screening mistaken for active disease (F5), or presumptive treatment without proof.
- **Archetype C (Syndromic Mimic / Ambiguity - DEER Class 3.2 & 4.1):** True clinical mimicry, multi-condition overlap, or viral bystander reactivation.

---

## Required Output Format for Each Case

```markdown
### Adjudication Case [Case #]: [Case ID]

#### 1. Clinical Forensic Scratchpad
- **True Underlying Pathology:** [Concise clinical diagnosis and pathophysiology]
- **Archetype Classification:** [Archetype A | Archetype B | Archetype C]
- **Diagnostic Traps Identified:** [Method X: None / F1-F6; Method Y: None / F1-F6]
- **Laboratory & Microbiological Reality:** [What objective tests actually showed]

#### 2. Comparative Scorecard
| Evaluation Dimension | Method X (MedCase) | Method Y (12-Step) | Forensic Comment |
|:---|:---:|:---:|:---|
| 1. Diagnostic Accuracy & Specificity | [1-5] | [1-5] | [Brief critique] |
| 2. Rule-Out Rigor vs False Rule-In | [1-5] | [1-5] | [Brief critique] |
| 3. Temporality & Discourse Context | [1-5] | [1-5] | [Brief critique] |
| 4. Evidence Grounding & Fidelity | [1-5] | [1-5] | [Brief critique] |
| 5. Patient Safety & Prudence | [1-5] | [1-5] | [Brief critique] |
| **OVERALL TOTAL** | **/25** | **/25** | |

#### 3. Clinical Forensic Critique
- **Method X Failure/Success Analysis:** [Detailed breakdown of MedCase's diagnostic trajectory]
- **Method Y Failure/Success Analysis:** [Detailed breakdown of 12-Step's evidentiary grounding]

#### 4. Final Judicial Verdict
- **Adjudicated Winner:** **[METHOD_X | METHOD_Y | TIE_BOTH_VALID | NEITHER_VALID]**
- **Definitive Medical Rationale:** [Clear, authoritative evidence-based justification]
- **Prevented Clinical Harm:** [Explanation of what medical harm is prevented by following the winning method]
```
"""


def format_deepseek_case_entry(
    idx: int, 
    r_mc: Dict[str, Any], 
    pair_index: Dict[Tuple[str, str], Dict[str, Any]], 
    all_anns: List[Dict[str, Any]]
) -> str:
    """Formats a single case for DeepSeek evaluation.
    Strictly Double-Blind: No hardcoded archetype, no leading forensic hints.
    """
    cid = r_mc.get("case_id", "")
    cid_base = cid.split("__")[0].strip()
    td = r_mc.get("target_disease", "")
    cl = r_mc.get("clinical_label", "")
    fd = r_mc.get("final_diagnosis", "")
    prompt = r_mc.get("case_prompt", "").strip()
    analysis = r_mc.get("case_analysis", "").strip()
    reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()
    
    r_12_hyp = pair_index.get((cid_base, normalize_disease_key(td)))
    
    # Resolve predicted entity using set tokenization if direct lookup returns None
    r_12_pred = pair_index.get((cid_base, normalize_disease_key(fd)))
    if not r_12_pred:
        for pe in parse_disease_entities(fd):
            if (cid_base, pe) in pair_index:
                r_12_pred = pair_index[(cid_base, pe)]
                break
    
    label_hyp = r_12_hyp.get("step12_dataset_label", cl) if r_12_hyp else cl
    role_hyp = r_12_hyp.get("step8_clinical_role", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    ev_str_hyp = r_12_hyp.get("step9_evidence_strength", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    rationale_hyp = r_12_hyp.get("rationale", "") if r_12_hyp else ""
    
    label_pred = r_12_pred.get("step12_dataset_label", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    role_pred = r_12_pred.get("step8_clinical_role", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    ev_str_pred = r_12_pred.get("step9_evidence_strength", "NOT_FORMALLY_EVALUATED") if r_12_pred else "NOT_FORMALLY_EVALUATED"
    rationale_pred = r_12_pred.get("rationale", "") if r_12_pred else ""
    
    lines = []
    lines.append(f"### Case {idx}: `{cid}`")
    lines.append("")
    lines.append(f"- **Target Hypothesis Audited:** `{td}` | **Ground Truth Label:** `{cl}`")
    lines.append(f"- **Method X (MedCase GPT-OSS 120B) Selected Diagnosis:** **`{fd}`**")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Status for `{td}`:** **`{label_hyp}`** (Role: `{role_hyp}`, Strength: `{ev_str_hyp}`)")
    lines.append(f"- **Method Y Status for `{fd}` (Predicted Entity):** **`{label_pred}`** (Role: `{role_pred}`)")
    lines.append("")
    lines.append("#### 1. Pre-Diagnostic Clinical Presentation (`case_prompt`)")
    lines.append("> " + prompt.replace("\n", "\n> "))
    lines.append("")
    lines.append("#### 2. Method X: Stanford MedCaseReasoning (GPT-OSS 120B Trajectory)")
    lines.append(f"- **Primary Diagnostic Choice:** **`{fd}`**")
    lines.append("- **Breakpoint & Scope Analysis:**")
    lines.append("```text")
    lines.append(analysis)
    lines.append("```")
    lines.append("- **Differential Diagnostic Reasoning Steps:**")
    lines.append(reasoning_mc)
    lines.append("")
    lines.append("#### 3. Method Y: MultiCaRe 12-Step Clinical Annotation (Evidentiary Audit)")
    lines.append(f"- **Adjudication for `{td}`:** **`{label_hyp}`** (Role: `{role_hyp}` | Evidence: `{ev_str_hyp}`)")
    lines.append(f"- **Adjudication for `{fd}`:** **`{label_pred}`** (Role: `{role_pred}` | Evidence: `{ev_str_pred}`)")
    lines.append("- **Direct Evidence Extracted for Hypothesis:**")
    lines.append(format_evidence_block(r_12_hyp))
    if r_12_pred and r_12_pred != r_12_hyp:
        lines.append(f"- **Direct Evidence Extracted for Predicted Entity (`{fd}`):**")
        lines.append(format_evidence_block(r_12_pred))
    lines.append("- **Clinical Rationale:**")
    lines.append(rationale_hyp or rationale_pred or "Evidence and diagnostic criteria evaluated against standard protocol.")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


def generate_deepseek_prompt(
    group3: List[Any], 
    pair_index: Dict[Tuple[str, str], Dict[str, Any]], 
    all_by_case: Dict[str, List[Dict[str, Any]]], 
    split_name: str
) -> str:
    """Generates the complete DeepSeek evaluation markdown benchmark file."""
    n_cases = len(group3)
    lines = []
    lines.append(build_deepseek_header(n_cases, split_name))
    lines.append(f"## Executive Summary Table ({n_cases} Cases)")
    lines.append("")
    lines.append("| # | Case ID | Target Disease | Clinical Ground Truth | Method X (MedCase) Prediction | Method Y (12-Step) Status |")
    lines.append("|:---:|:---|:---:|:---:|:---|:---|")
    
    for idx, (r_mc, all_anns) in enumerate(group3, 1):
        cid = r_mc.get("case_id", "")
        cid_base = cid.split("__")[0].strip()
        td = r_mc.get("target_disease", "")
        cl = r_mc.get("clinical_label", "")
        fd = r_mc.get("final_diagnosis", "")
        
        r_12_pred = pair_index.get((cid_base, normalize_disease_key(fd)))
        if not r_12_pred:
            for pe in parse_disease_entities(fd):
                if (cid_base, pe) in pair_index:
                    r_12_pred = pair_index[(cid_base, pe)]
                    break
                    
        lbl_pred = r_12_pred.get("step12_dataset_label", "NOT_ANNOTATED") if r_12_pred else "NOT_ANNOTATED"
        lines.append(f"| {idx} | `{cid}` | {td} | **{cl}** | **`{fd}`** | `{lbl_pred}` |")
        
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## Case Dossiers for Adjudication")
    lines.append("")
    
    for idx, (r_mc, all_anns) in enumerate(group3, 1):
        lines.append(format_deepseek_case_entry(idx, r_mc, pair_index, all_anns))
        
    return "\n".join(lines)

## 7. Batch Pipeline Orchestrator Across All 5 GPT-OSS Splits

The batch orchestrator automates the entire processing pipeline across all 5 mismatch splits:
1. Iterates over all `.jsonl` files in `codebases/data_processing/data/mismacht-gpt-oss/*.jsonl`.
2. Creates an isolated output directory `codebases/data_processing/data/mismacht-gpt-oss/outputs/<split_name>/`.
3. Writes the complete set of 5 standardized files per split:
   - `group1_cross_disease_agreement.md`
   - `group2_out_of_scope.md`
   - `group3_diagnostic_divergence.md`
   - `tripartite_summary.json`
   - `claude_benchmark_prompt.md`
   - `deepseek_benchmark_prompt.md`
4. Aggregates all splits into a unified master file:
   - `codebases/data_processing/data/mismacht-gpt-oss/outputs/consolidated_tripartite_summary.json`.

In [8]:
def process_single_split(
    mismatch_path: Path, 
    output_base_dir: Path, 
    annotated_map: Dict[Tuple[str, str], Dict[str, Any]], 
    annotated_by_case: Dict[str, List[Dict[str, Any]]]
) -> Dict[str, Any]:
    """Processes one mismatch file, generating all reports, prompt benchmarks, and summary JSON."""
    split_name = mismatch_path.stem
    split_out_dir = output_base_dir / split_name
    split_out_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Load records
    with open(mismatch_path, "r", encoding="utf-8") as fp:
        records = [json.loads(line) for line in fp if line.strip()]
        
    total_recs = len(records)
    print(f"\n--- Processing Split: {split_name} ({total_recs} records) ---")
    
    # 2. Partition into 3 groups
    group1, group2, group3 = partition_records(records, annotated_map, annotated_by_case)
    print(f"  -> Group 1 (Multi-Pathogen Agreement): {len(group1)} ({len(group1)/total_recs*100:.1f}%)")
    print(f"  -> Group 2 (Out-of-Scope Integrity):   {len(group2)} ({len(group2)/total_recs*100:.1f}%)")
    print(f"  -> Group 3 (Genuine Divergence):       {len(group3)} ({len(group3)/total_recs*100:.1f}%)")
    
    # 3. Build & Write Reports
    rep1_md = build_report_group1(group1, total_recs, split_name)
    with open(split_out_dir / "group1_cross_disease_agreement.md", "w", encoding="utf-8") as f:
        f.write(rep1_md)
        
    rep2_md = build_report_group2(group2, total_recs, split_name)
    with open(split_out_dir / "group2_out_of_scope.md", "w", encoding="utf-8") as f:
        f.write(rep2_md)
        
    rep3_md = build_report_group3(group3, total_recs, split_name, annotated_map)
    with open(split_out_dir / "group3_diagnostic_divergence.md", "w", encoding="utf-8") as f:
        f.write(rep3_md)
        
    # 4. Export Summary JSON
    summary_dict = build_tripartite_summary_json(total_recs, group1, group2, group3, split_name)
    with open(split_out_dir / "tripartite_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary_dict, f, indent=2, ensure_ascii=False)
        
    # 5. Build Judicial Prompts (if Group 3 has cases)
    if group3:
        claude_md = generate_claude_prompt(group3, annotated_map, annotated_by_case, split_name)
        with open(split_out_dir / "claude_benchmark_prompt.md", "w", encoding="utf-8") as f:
            f.write(claude_md)
            
        deepseek_md = generate_deepseek_prompt(group3, annotated_map, annotated_by_case, split_name)
        with open(split_out_dir / "deepseek_benchmark_prompt.md", "w", encoding="utf-8") as f:
            f.write(deepseek_md)
            
    print(f"  ✓ Output written to {split_out_dir}")
    return summary_dict


def run_batch_pipeline(
    mismatch_dir: Path, 
    output_base_dir: Path, 
    annotated_map: Dict[Tuple[str, str], Dict[str, Any]], 
    annotated_by_case: Dict[str, List[Dict[str, Any]]]
) -> Dict[str, Any]:
    """Runs the complete batch pipeline across all mismatch jsonl files."""
    output_base_dir.mkdir(parents=True, exist_ok=True)
    mismatch_files = sorted(list(mismatch_dir.glob("*.jsonl")))
    if not mismatch_files:
        raise FileNotFoundError(f"No jsonl files found in mismatch directory: {mismatch_dir}")
        
    split_summaries: List[Dict[str, Any]] = []
    total_cohort_cases = 0
    total_g1 = 0
    total_g2 = 0
    total_g3 = 0
    
    for mf in mismatch_files:
        summary = process_single_split(mf, output_base_dir, annotated_map, annotated_by_case)
        split_summaries.append(summary)
        total_cohort_cases += summary["total_cases"]
        total_g1 += summary["group1_multi_pathogen_agreement"]["count"]
        total_g2 += summary["group2_out_of_scope_integrity"]["count"]
        total_g3 += summary["group3_genuine_diagnostic_divergence"]["count"]
        
    consolidated_summary = {
        "master_title": "MultiCaRe Consolidated Tripartite Cross-Validation Master Summary",
        "description": "Cross-validation partitioning of Stanford MedCaseReasoning (GPT-OSS 120B) vs. MultiCaRe 12-Step pipeline discrepancies across all 5 candidate splits.",
        "splits_evaluated_count": len(split_summaries),
        "total_cohort_cases": total_cohort_cases,
        "aggregate_metrics": {
            "group1_multi_pathogen_agreement": {
                "count": total_g1,
                "percentage": round(total_g1 / total_cohort_cases * 100, 2) if total_cohort_cases else 0.0,
                "clinical_interpretation": "Both systems agree on confirmed positive conditions; divergence was an artifact of multiclass single-selection vs multi-disease hypothesis testing."
            },
            "group2_out_of_scope_integrity": {
                "count": total_g2,
                "percentage": round(total_g2 / total_cohort_cases * 100, 2) if total_cohort_cases else 0.0,
                "clinical_interpretation": "MedCase correctly recognized unlisted/rare condition as OUT_OF_SCOPE, preserving closed-taxonomy integrity."
            },
            "group3_genuine_diagnostic_divergence": {
                "count": total_g3,
                "percentage": round(total_g3 / total_cohort_cases * 100, 2) if total_cohort_cases else 0.0,
                "clinical_interpretation": "Genuine diagnostic conflicts, rule-in vs rule-out tensions, or unresolved syndromic mimics routed to LLM judicial arbiters."
            }
        },
        "per_split_breakdown": split_summaries
    }
    
    master_path = output_base_dir / "consolidated_tripartite_summary.json"
    with open(master_path, "w", encoding="utf-8") as f:
        json.dump(consolidated_summary, f, indent=2, ensure_ascii=False)
        
    print(f"\n=======================================================")
    print(f"BATCH EXECUTION COMPLETE!")
    print(f"Master Consolidated Summary written to: {master_path}")
    print(f"=======================================================\n")
    return consolidated_summary

## 8. Live Execution & Consolidated Scorecard Display

Execute the batch orchestrator across all 5 mismatch partitions and visualize the consolidated tripartite distribution table.

In [9]:
# Execute complete batch pipeline
master_summary = run_batch_pipeline(MISMATCH_DIR, OUTPUT_BASE_DIR, annotated_map, annotated_by_case)

# Display tabular executive summary
agg = master_summary["aggregate_metrics"]
total = master_summary["total_cohort_cases"]

if RICH_AVAILABLE:
    console = Console()
    table = Table(
        title="MultiCaRe Tripartite Cross-Validation Consolidated Scorecard (591 Cases)",
        box=box.ROUNDED,
        header_style="bold cyan",
        title_style="bold magenta"
    )
    table.add_column("Cohort Split", style="bold white", width=26)
    table.add_column("Total", justify="right", style="cyan")
    table.add_column("Group 1 (Agreement)", justify="right", style="green")
    table.add_column("Group 2 (Out-of-Scope)", justify="right", style="yellow")
    table.add_column("Group 3 (Divergence)", justify="right", style="red")
    
    for sp in master_summary["per_split_breakdown"]:
        table.add_row(
            sp["split_name"],
            str(sp["total_cases"]),
            f"{sp['group1_multi_pathogen_agreement']['count']} ({sp['group1_multi_pathogen_agreement']['percentage']:.1f}%)",
            f"{sp['group2_out_of_scope_integrity']['count']} ({sp['group2_out_of_scope_integrity']['percentage']:.1f}%)",
            f"{sp['group3_genuine_diagnostic_divergence']['count']} ({sp['group3_genuine_diagnostic_divergence']['percentage']:.1f}%)"
        )
    
    table.add_section()
    table.add_row(
        "TOTAL CONSOLIDATED",
        str(total),
        f"{agg['group1_multi_pathogen_agreement']['count']} ({agg['group1_multi_pathogen_agreement']['percentage']:.1f}%)",
        f"{agg['group2_out_of_scope_integrity']['count']} ({agg['group2_out_of_scope_integrity']['percentage']:.1f}%)",
        f"{agg['group3_genuine_diagnostic_divergence']['count']} ({agg['group3_genuine_diagnostic_divergence']['percentage']:.1f}%)",
        style="bold magenta"
    )
    console.print(table)
else:
    print("""
+--------------------------------------------------------------------------------------------------+
| MULTICARE TRIPARTITE CROSS-VALIDATION CONSOLIDATED SCORECARD                                     |
+--------------------------------------------------------------------------------------------------+
| Cohort Split              | Total Cases | Group 1 (Agreement) | Group 2 (Out-of-Scope) | Group 3 (Divergence) |
+---------------------------+-------------+---------------------+------------------------+----------------------+""")

    for sp in master_summary["per_split_breakdown"]:
        s_name = sp["split_name"]
        s_tot = sp["total_cases"]
        g1 = f"{sp['group1_multi_pathogen_agreement']['count']} ({sp['group1_multi_pathogen_agreement']['percentage']:.1f}%)"
        g2 = f"{sp['group2_out_of_scope_integrity']['count']} ({sp['group2_out_of_scope_integrity']['percentage']:.1f}%)"
        g3 = f"{sp['group3_genuine_diagnostic_divergence']['count']} ({sp['group3_genuine_diagnostic_divergence']['percentage']:.1f}%)"
        print(f"| {s_name:<25} | {s_tot:<11} | {g1:<19} | {g2:<22} | {g3:<20} |")

    print("+---------------------------+-------------+---------------------+------------------------+----------------------+")
    tot_g1 = f"{agg['group1_multi_pathogen_agreement']['count']} ({agg['group1_multi_pathogen_agreement']['percentage']:.1f}%)"
    tot_g2 = f"{agg['group2_out_of_scope_integrity']['count']} ({agg['group2_out_of_scope_integrity']['percentage']:.1f}%)"
    tot_g3 = f"{agg['group3_genuine_diagnostic_divergence']['count']} ({agg['group3_genuine_diagnostic_divergence']['percentage']:.1f}%)"
    print(f"| TOTAL CONSOLIDATED        | {total:<11} | {tot_g1:<19} | {tot_g2:<22} | {tot_g3:<20} |")
    print("+--------------------------------------------------------------------------------------------------+")

print("\nGenerated Outputs Manifest:")
for p in sorted(OUTPUT_BASE_DIR.glob("**/*")):
    if p.is_file():
        rel = p.relative_to(MISMATCH_DIR)
        print(f" - {rel} ({p.stat().st_size:,} bytes)")


--- Processing Split: unmatched_part_1_Khai (119 records) ---
  -> Group 1 (Multi-Pathogen Agreement): 47 (39.5%)
  -> Group 2 (Out-of-Scope Integrity):   54 (45.4%)
  -> Group 3 (Genuine Divergence):       18 (15.1%)
  ✓ Output written to /Users/Me/Desktop/Khailambaitest/final_year/chatbot/project/aiRepo/HealthCareAgent/data/mismacht-gpt-oss/outputs/unmatched_part_1_Khai

--- Processing Split: unmatched_part_2_Khoi (118 records) ---
  -> Group 1 (Multi-Pathogen Agreement): 49 (41.5%)
  -> Group 2 (Out-of-Scope Integrity):   58 (49.2%)
  -> Group 3 (Genuine Divergence):       11 (9.3%)
  ✓ Output written to /Users/Me/Desktop/Khailambaitest/final_year/chatbot/project/aiRepo/HealthCareAgent/data/mismacht-gpt-oss/outputs/unmatched_part_2_Khoi

--- Processing Split: unmatched_part_3_AMinh (118 records) ---
  -> Group 1 (Multi-Pathogen Agreement): 47 (39.8%)
  -> Group 2 (Out-of-Scope Integrity):   67 (56.8%)
  -> Group 3 (Genuine Divergence):       4 (3.4%)
  ✓ Output written to /Users/Me

---

# Group 2 Forensic Adjudication Extension (Stripped `OUT_OF_SCOPE` + Full `raw_case_text` + Archetype D + Dimension $D_6$)

## 9. Raw Case Text (`raw_case_text`) Ingestion & `OUT_OF_SCOPE` Stripping Utilities

To extend clinical LLM judicial arbitration to **Group 2 (`OUT_OF_SCOPE`)** while preserving the Group 3 implementation above for side-by-side comparison, this section introduces:
1. **Full Source Text Ingestion (`load_raw_case_text_database`):** Loads the complete, un-truncated `raw_case_text` from `data/soap/**/*.json` (structured according to `resources/prompts/soap/soap_structuring_prompt.md`), eliminating pre-breakpoint truncation artifacts in Dimension $D_4$ (`evidence_grounding`).
2. **`OUT_OF_SCOPE` Stripping & Scope Flag Extraction (`extract_stripped_diagnosis`):** Strips `-> OUT_OF_SCOPE` (and outer bracket formatting) to expose Method X's actual predicted disease entity for direct head-to-head clinical comparison against Method Y, while retaining the explicit `OUT_OF_SCOPE` flag and the 21-disease closed taxonomy list so the arbiter can also verify whether the out-of-scope refusal was valid or a false refusal of an in-scope disease.
3. **Archetype D (`ARCHETYPE_D` — Dual-System Validity):** Represents cases where both systems provide clinically valid diagnoses (e.g., Method X identifies an out-of-scope principal pathology or acute complication while Method Y confirms a true comorbid, historical, or secondary target infection). When Archetype D applies, both systems are marked as right (`BOTH_RIGHT`).
4. **Dimension 6 ($D_6$: `diagnostic_primacy` — Spanish ICD-10 Coding Manual / *Manual de Codificación CIE-10-ES Diagnósticos*):** Evaluates whether each method's diagnosis captures the **principal / definitive diagnosis** centrally discussed in the source case report (5), a real **secondary / comorbid / complicating condition** (3), or an unrelated/incorrect/colonization label (1).

In [10]:
SOAP_DIR = WORKSPACE_ROOT / "data/soap"

CLOSED_21_DISEASES = [
    "Tuberculosis", "HIV_AIDS", "Syphilis", "Leishmaniasis", "Dengue",
    "Malaria", "Brucellosis", "Cytomegalovirus", "Aspergillosis",
    "Echinococcosis", "Cryptococcosis", "Nocardiosis", "Pneumocystis",
    "COVID-19", "Histoplasmosis", "Leprosy", "Neurocysticercosis",
    "Cholera", "Ebola_Filovirus", "Rabies", "Yellow_Fever"
]


def load_raw_case_text_database(soap_dir: Path) -> Dict[str, str]:
    """Loads full verbatim raw_case_text from data/soap/**/*.json indexed by pair stem and base case_id."""
    raw_text_map: Dict[str, str] = {}
    if not soap_dir.exists():
        raise FileNotFoundError(f"SOAP directory not found: {soap_dir}")

    loaded_files = 0
    for fpath in sorted(soap_dir.glob("**/*.json")):
        if fpath.name == "manifest.json":
            continue
        with open(fpath, "r", encoding="utf-8") as fp:
            rec = json.loads(fp.read())
        raw_txt = (rec.get("raw_case_text") or "").strip()
        if not raw_txt:
            continue
        raw_text_map[fpath.stem] = raw_txt
        cid_base = (rec.get("case_id") or fpath.stem.split("__")[0]).strip()
        if cid_base:
            raw_text_map.setdefault(cid_base, raw_txt)
        loaded_files += 1

    print(f"Loaded raw_case_text from {loaded_files:,} SOAP JSON files ({len(raw_text_map):,} lookup keys).")
    return raw_text_map


def extract_stripped_diagnosis(fd: str) -> Tuple[str, str]:
    """Strips '-> OUT_OF_SCOPE' from MedCase final_diagnosis while preserving the scope flag.
    
    Returns:
        (stripped_disease_name, scope_flag)
    """
    raw = (fd or "").strip()
    has_oos = "out_of_scope" in raw.lower()
    stripped = re.sub(r"\s*->\s*out_of_scope.*$", "", raw, flags=re.IGNORECASE).strip()
    stripped = stripped.strip("[]").strip()
    scope_flag = "OUT_OF_SCOPE" if has_oos else "IN_SCOPE"
    return (stripped or raw, scope_flag)


def resolve_raw_case_text(r_mc: Dict[str, Any], raw_text_map: Dict[str, str]) -> str:
    """Resolves the full raw_case_text for a mismatch record, falling back to case_prompt only if missing."""
    cid = r_mc.get("case_id", "").strip()
    cid_base = cid.split("__")[0].strip()
    if cid in raw_text_map:
        return raw_text_map[cid]
    if cid_base in raw_text_map:
        return raw_text_map[cid_base]
    return r_mc.get("case_prompt", "").strip()


raw_case_text_map = load_raw_case_text_database(SOAP_DIR)

Loaded raw_case_text from 3,289 SOAP JSON files (6,091 lookup keys).


## 10. Group 2 Anthropic Claude Forensic Adjudication Prompt Generator

This generator mirrors Section 5 (Group 3 Claude Prompt Generator) with the four Group 2 enhancements:
- **Stripped `OUT_OF_SCOPE` Diagnosis + Scope Audit:** Evaluates Method X's stripped primary disease prediction directly while exposing its `OUT_OF_SCOPE` scope decision against the 21-disease list.
- **Full `raw_case_text` in `<clinical_case_presentation>`:** Replaces truncated `case_prompt` with full verbatim `raw_case_text` from `data/soap/`.
- **Archetype D (`ARCHETYPE_D`):** Dual-system validity when both Method X (stripped disease) and Method Y (12-step target hypothesis) identify real, clinically valid conditions present in the patient; mandates marking both as right (`BOTH_RIGHT`).
- **Dimension 6 ($D_6$: `diagnostic_primacy`):** Grounded in the **Spanish ICD-10 Coding Manual (*Manual de Codificación CIE-10-ES Diagnósticos*)**, scoring whether the predicted diagnosis is the principal/definitive diagnosis of the case (5), a real secondary/comorbid/complicating condition (3), or unrelated/incorrect/colonization (1).

In [11]:
def build_claude_group2_system_prompt() -> str:
    """Constructs Anthropic Claude System Prompt for Group 2 evaluation with Archetype D and Dimension D6."""
    closed_21_str = ", ".join(CLOSED_21_DISEASES)
    return f"""<system_instructions>

<persona>
You are a Senior Clinical Forensic Arbiter — an elite physician-scientist with dual board certification in
Infectious Disease and Clinical Pathology, and extensive expertise in Diagnostic Decision Systems, ICD-10 Clinical Coding, and Evidence-Based Medicine.
You chair hospital Diagnostic Quality Assurance Boards and specialize in resolving high-stakes clinical diagnostic evaluations
between autonomous clinical reasoning algorithms. Your adjudications are rigorous, evidence-grounded, and defensible at grand rounds.
</persona>

<analytical_stance>
You are strictly EVIDENCE-FIRST, not narrative-first. Diagnostic intuition, syndromic plausibility, and travel history are
hypothesis generators, NEVER substitutes for objective microbiological or histopathological verification.

In this Group 2 cohort, Method X (Stanford MedCaseReasoning, GPT-OSS 120B) originally appended `-> OUT_OF_SCOPE` to its predicted diagnosis,
claiming the patient's principal diagnosis lay outside its closed 21-disease taxonomy ({closed_21_str}).
We have stripped the `-> OUT_OF_SCOPE` suffix so you can directly evaluate Method X's predicted disease entity alongside its scope decision,
comparing it against Method Y (MultiCaRe 12-Step Clinical Annotation Pipeline) on the full verbatim clinical case report (`raw_case_text`).

Under evidence-based diagnostic heuristics (Sackett et al., Evidence-Based Medicine 2000; Schiff et al., Arch Intern Med 2009):
1. Definitive negative gold-standard laboratory testing (negative smears, negative cultures, non-reactive specific serology/PCR,
   or contradicting biopsy findings) conclusively rules OUT the disease under SpPIn/SnNOut principles. Affirming an excluded diagnosis
   constitutes an unacceptable breach of evidence-based clinical reasoning.
2. Inactive past medical history cured years/decades prior (F4) MUST NOT be affirmed as the principal etiology of an acute current illness
   (Uzuner et al., JAMIA 2011).
3. Prophylactic interventions, vaccination scars (e.g. BCG scar), and negative screening history (F5) MUST NOT be confused
   with active pathologic infection (Harkema et al., JBI 2009).
4. Presumptive or empiric clinical therapy without laboratory confirmation (F6) must be annotated as UNRESOLVED / PRESUMPTIVE,
   never as an established definitive gold-standard positive (CDC MMWR Case Definitions 1997).
5. Under the Spanish ICD-10 Coding Manual (Manual de Codificación CIE-10-ES Diagnósticos), two methods can both name real diseases present
   in the patient (Archetype D), yet only one may capture the principal / definitive diagnosis chiefly responsible for the clinical episode (D6).

You aggressively detect and penalize:
- FALSE POSITIVE RULE-IN errors (DEER Class 2.4/3.2): Affirming a diagnosis despite definitive negative lab testing.
- FALSE OUT-OF-SCOPE REFUSALS: Flagging `OUT_OF_SCOPE` when the stripped predicted diagnosis actually belongs to the 21-disease taxonomy.
- TEMPORALITY CONFUSION (DEER Class 3.2 / F4): Confusing cured historical disease with current acute presentation.
- DISCOURSE SCOPE DRIFT (DEER Class 2.4 / F5): Confusing vaccines/prophylaxis with active infection.
- VERBOSITY BIAS: Do not reward verbose reasoning traces that arrive at an ungrounded or contradicted conclusion.
</analytical_stance>

<epistemic_literature_grounding>
Your judicial evaluation must strictly embody peer-reviewed diagnostic error, clinical coding, and clinical AI consensus:
- Dimension 1 (Diagnostic Accuracy, D1): Grounded in Singhal et al. (Med-PaLM, Nature 2023) and Tu et al. (AMIE, Nature 2025). Evaluates identification of true etiologic pathogen/pathology confirmed by CDC/WHO or gold-standard diagnostic criteria [E1/E2].
- Dimension 2 (Rule-Out Rigor, D2): Grounded in Sackett et al. (EBM 2000; SpPIn/SnNOut) and Schiff et al. (DEER Taxonomy, Arch Intern Med 2009). Disregarding definitive negative laboratory or biopsy testing constitutes a catastrophic diagnostic error (score <= 2).
- Dimension 3 (Temporality & Context, D3): Grounded in Uzuner et al. (i2b2 Assertion, JAMIA 2011) and Harkema et al. (ConText, JBI 2009). Demands strict discrimination between acute pathology vs cured past history [F4] or prophylactic vaccination/screening [F5].
- Dimension 4 (Evidence Grounding, D4): Grounded in Goyal et al. / CliniFact (JAMIA 2025). Demands strict factual alignment between generated clinical claims and the full source case text (`raw_case_text`), heavily penalizing hallucinated labs or findings.
- Dimension 5 (Patient Safety, D5): Grounded in Singh & Graber (Diagnosis 2014) and Med-PaLM Harm Stratification Matrix (Nature 2023). Penalizes preventable patient harm, toxic unneeded therapy cascades, and missed life-threatening infections.
- Dimension 6 (Diagnostic Primacy, D6): Grounded in the Spanish ICD-10 Coding Manual (Manual de Codificación CIE-10-ES Diagnósticos, Ministerio de Sanidad). Distinguishes the principal / definitive diagnosis (Diagnóstico Principal) chiefly responsible for the patient's admission and central discussion from secondary, comorbid, or complicating conditions (Diagnósticos Secundarios).
</epistemic_literature_grounding>

<evidence_grading_framework>
You apply the MultiCaRe Evidence Hierarchy grounded in CDC MMWR Case Definitions (1997) and WHO surveillance criteria in all assessments:
- E1_CONFIRMATORY_LAB: Definitive gold-standard pathogen/pathology proof (culture, PCR, biopsy histopathology, direct visualization).
- E2_HIGHLY_SPECIFIC: Highly specific validated assays (specific antigen EIA, IgM seroconversion, IgG avidity).
- E3_SUGGESTIVE_NON_SPECIFIC: Clinical syndrome, non-specific lab trends, imaging patterns — suggestive but INSUFFICIENT alone for definitive confirmation.
- E0_CONTRADICTED: Objective testing performed and explicitly NEGATIVE — disease is RULED OUT unless proven false-negative.
- E0_NO_EVIDENCE: Diagnostic testing absent or non-confirmatory — absence of evidence is not evidence of disease.
- E4_CONTEXT: Mentions strictly relating to background, differential discussion, cured history, or vaccination (F4, F5).
</evidence_grading_framework>

<dilemma_archetypes>
Classify each case into one of four canonical clinical archetypes:
- ARCHETYPE_A (Confirmation Bias & Rule-Out Overwrite / Temporality - DEER Class 3.2): One system anchors on syndromic presentation while overriding negative confirmatory tests or confusing remote cured past history (F4) with acute active disease.
- ARCHETYPE_B (Contextual Misattribution & Presumptive Closure - DEER Class 2.4 & 3.2): One system misattributes prophylactic vaccination, screening (F5), or empiric therapy without microbiological proof (F6) as confirmed active disease.
- ARCHETYPE_C (Syndromic Look-Alike & Complex Differential Ambiguity - DEER Class 3.2 & 4.1): True clinical mimicry, overlapping syndromic presentation, or false OUT_OF_SCOPE refusal on an in-scope target disease.
- ARCHETYPE_D (Dual-System Validity — Principal vs. Comorbid/Secondary Complementarity): Both systems provide clinically valid answers! For example, Method X's stripped diagnosis accurately identifies the principal out-of-scope pathology or acute drug/treatment complication, while Method Y accurately verifies a true comorbid, underlying, or secondary target infection genuinely present in the patient (or vice versa). When ARCHETYPE_D applies, you MUST mark both systems as right (`winner: "BOTH_RIGHT"`), and use Dimension 6 (`D6: diagnostic_primacy`) to score which method captured the centrally discussed / definitive principal diagnosis (5) versus a secondary/comorbid condition (3).
</dilemma_archetypes>

<anti_sycophancy_directives>
The following are HARD PROHIBITIONS — violating any of these constitutes an adjudication failure:
- PROHIBITION 1: NEVER declare a diplomatic tie when one method commits an actual diagnostic or evidentiary error (F1-F6); HOWEVER, when BOTH systems give genuinely valid answers for conditions present in the patient (ARCHETYPE_D), you MUST mark `winner: "BOTH_RIGHT"` and differentiate their principal vs. secondary role via Dimension D6.
- PROHIBITION 2: NEVER accept narrative plausibility over negative laboratory results. If a test is explicitly negative, any method claiming that disease is POSITIVE has committed an egregious diagnostic failure (Schiff et al. 2009).
- PROHIBITION 3: NEVER penalize Method Y for correctly annotating a disease as NEGATIVE simply because the patient presented with symptoms resembling that disease.
- PROHIBITION 4: NEVER confuse historical resolved conditions (e.g., childhood TB cured 20 years ago) with active presenting diagnoses (Uzuner et al. 2011; Harkema et al. 2009).
</anti_sycophancy_directives>

<scoring_rubric>
Score each method independently on 6 dimensions (1 to 5 integer scale; total out of 30):

D1: diagnostic_accuracy (1-5)
  5: Accurately identifies a true etiologic or comorbid condition present in the patient backed by definitive gold-standard evidence (E1/E2).
  3: Plausible differential or valid condition with incomplete specificity or minor scope error (e.g., false OUT_OF_SCOPE flag on an in-scope disease).
  1: Entirely incorrect, hallucinated diagnosis, or contradicts confirmed laboratory reality.

D2: rule_out_rigor (1-5)
  5: Flawlessly respects negative laboratory/biopsy findings; correctly excludes non-viable candidates.
  3: Incomplete exclusion; leaves ruled-out conditions ambiguous.
  1: Catastrophic false-positive rule-in; affirms disease directly contradicted by negative lab/pathology.

D3: temporality_context (1-5)
  5: Perfectly differentiates active acute illness from remote history (F4) and prophylactic exposure (F5).
  3: Minor ambiguity in temporal anchoring but reaches acceptable conclusion.
  1: Gross conflation of remote past history or vaccination with acute active pathology.

D4: evidence_grounding (1-5)
  5: Every assertion is directly anchored in documented facts from `raw_case_text`; zero hallucination.
  3: Mostly grounded but includes unverified clinical extrapolations.
  1: Severe hallucination of clinical facts, tests, or results not in the source text.

D5: patient_safety (1-5)
  5: Adjudication leads to correct, targeted management; minimizes unnecessary toxicity.
  3: Suboptimal management path with moderate risk of delayed treatment or overtreatment.
  1: Dangerous failure; triggers toxic unneeded therapies or misses lethal disease.

D6: diagnostic_primacy (1-5) [Grounded in Spanish ICD-10 Coding Manual / Manual de Codificación CIE-10-ES Diagnósticos]
  Two methods can both name real diseases, yet only one may capture the centrally discussed / definitive diagnosis of the case:
  5: Predicted label is the principal / definitive diagnosis that the source case identifies as the main disease (e.g., GLA over aspergillosis; ABPM over aspergillosis alone; IgA-PIGN over aspergillosis).
  3: Predicted label is a real but secondary / comorbid / complicating condition that is explicitly present but not the principal diagnosis (e.g., calling the rheumatoid arthritis case “aspergillosis” when aspergilloma is past history; calling the CVID case “aspergillosis” when the admission is voriconazole photosensitivity).
  1: Predicted label is unrelated, incorrect, or a trivial colonisation/contaminant presented as the main disease.
</scoring_rubric>

<mandatory_workflow>
For each case, you MUST execute your analysis sequentially inside `<clinical_scratchpad>` before producing the final JSON verdict:
Step 1: Clinical Fact & Principal Diagnosis Extraction — From the full `raw_case_text`, extract all objective laboratory/histopathology results and identify the principal diagnosis vs. secondary/comorbid conditions under Spanish ICD-10 coding criteria.
Step 2: Method X (MedCase Stripped Diagnosis & Scope) Audit — Evaluate Method X's stripped predicted diagnosis (`stripped_predicted_diagnosis`) for clinical truth and diagnostic primacy (D6), and check whether its `OUT_OF_SCOPE` flag was valid or a false refusal of one of the 21 diseases.
Step 3: Method Y (12-Step) Audit — Evaluate whether Method Y's label is supported by the MultiCaRe evidence hierarchy and whether the target disease represents the principal diagnosis (D6=5), a true secondary/comorbid condition (D6=3), or an erroneous label (D6=1, F1-F6).
Step 4: Tension Archetype Classification — Classify as ARCHETYPE_A, ARCHETYPE_B, ARCHETYPE_C, or ARCHETYPE_D (when both systems have valid answers).
Step 5: Forensic Synthesis — If ARCHETYPE_D applies, mark `winner: "BOTH_RIGHT"` and highlight which method captured higher diagnostic primacy (D6); otherwise declare the single superior method (`METHOD_X` or `METHOD_Y`).
</mandatory_workflow>

<output_format>
Your final output for each case MUST be valid JSON enclosed in `<forensic_adjudication>` tags matching this schema:
{{
  "case_id": "string",
  "dilemma_archetype": "ARCHETYPE_A | ARCHETYPE_B | ARCHETYPE_C | ARCHETYPE_D",
  "method_x_audit": {{
    "stripped_predicted_diagnosis": "string",
    "raw_output_with_scope": "string",
    "scope_flag_validity": "VALID_OUT_OF_SCOPE | FALSE_OUT_OF_SCOPE_REFUSAL",
    "clinical_status": "CONFIRMED_PRINCIPAL | CONFIRMED_SECONDARY_COMORBID | FALSE_POSITIVE_RULE_IN | HISTORICAL_CONFLATION | PROPHYLAXIS_CONFUSION | PLAUSIBLE_BUT_UNPROVEN | INCORRECT",
    "failure_modes": ["F1_LEXICAL_HOMONYM | F2_DOCUMENT_SCOPE | F3_PATIENT_ANCHORING | F4_TEMPORALITY_BIAS | F5_DISCOURSE_CONTEXT | F6_SPECIFICITY_GAP"],
    "scores": {{"D1": int, "D2": int, "D3": int, "D4": int, "D5": int, "D6": int, "total": int}}
  }},
  "method_y_audit": {{
    "evaluated_hypotheses": [{{"disease": "string", "label": "POSITIVE | NEGATIVE", "verdict_soundness": "CORRECT_PRINCIPAL | CORRECT_SECONDARY_COMORBID | ERRONEOUS"}}],
    "failure_modes": [],
    "scores": {{"D1": int, "D2": int, "D3": int, "D4": int, "D5": int, "D6": int, "total": int}}
  }},
  "judicial_verdict": {{
    "winner": "METHOD_X | METHOD_Y | BOTH_RIGHT | NEITHER_CORRECT",
    "diagnostic_primacy_holder": "METHOD_X | METHOD_Y | BOTH | NEITHER",
    "definitive_explanation": "string (evidence-based justification citing specific lab findings and principal vs secondary diagnosis analysis)",
    "prevented_clinical_harm": "string (what clinical harm was avoided or how both diagnoses complement patient care)"
  }}
}}
</output_format>

Think the problem through before you answer.
</system_instructions>"""


def build_claude_group2_user_header(
    batch_cases_count: int,
    batch_num: int,
    total_batches: int,
    start_idx: int,
    end_idx: int,
    total_split_cases: int,
    split_name: str
) -> str:
    """Constructs the user prompt header for a single 5-case Claude Group 2 batch block."""
    return f"""<evaluation_task split="{split_name}" batch="{batch_num}_of_{total_batches}" cases="{start_idx}_to_{end_idx}_of_{total_split_cases}">
<task_overview>
You are adjudicating Batch {batch_num} of {total_batches} ({batch_cases_count} cases: Cases {start_idx}–{end_idx} out of {total_split_cases} Group 2 `OUT_OF_SCOPE` cases in `{split_name}`) comparing:
- Method X: Stanford MedCaseReasoning (GPT-OSS 120B) — We stripped `-> OUT_OF_SCOPE` to expose its predicted primary disease name alongside its scope flag.
- Method Y: MultiCaRe 12-Step Clinical Annotation Pipeline — Targeted hypothesis verification engine.

For each of the {batch_cases_count} cases in this batch:
1. Read the full verbatim clinical case report (`raw_case_text`) inside `<clinical_case_presentation>`.
2. Conduct an unconstrained 5-step clinical forensic analysis inside `<clinical_scratchpad>`.
3. Apply the 6-dimensional scoring rubric (D1 to D6, 1-5 integer scale, where D6 evaluates Diagnostic Primacy under the Spanish ICD-10 Coding Manual).
4. Classify the case into `ARCHETYPE_A`, `ARCHETYPE_B`, `ARCHETYPE_C`, or `ARCHETYPE_D` (if both systems have valid answers, mark `winner: "BOTH_RIGHT"`).
5. Deliver your structured JSON verdict inside `<forensic_adjudication>`.
</task_overview>"""


def format_claude_group2_case_entry(
    idx: int,
    r_mc: Dict[str, Any],
    pair_index: Dict[Tuple[str, str], Dict[str, Any]],
    all_by_case: Dict[str, List[Dict[str, Any]]],
    raw_text_map: Dict[str, str]
) -> str:
    """Formats a single Group 2 case into Anthropic XML case tags using stripped OUT_OF_SCOPE diagnosis and full raw_case_text."""
    cid = r_mc.get("case_id", "")
    cid_base = cid.split("__")[0].strip()
    td = r_mc.get("target_disease", "")
    cl = r_mc.get("clinical_label", "")
    fd_raw = r_mc.get("final_diagnosis", "").strip()
    stripped_fd, scope_flag = extract_stripped_diagnosis(fd_raw)
    raw_case_text = resolve_raw_case_text(r_mc, raw_text_map)
    analysis = r_mc.get("case_analysis", "").strip()
    reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()

    all_anns = all_by_case.get(cid_base, [])
    r_12_hyp = pair_index.get((cid_base, normalize_disease_key(td)))

    # Resolve stripped predicted entity in 12-step annotations if evaluated
    r_12_pred = pair_index.get((cid_base, normalize_disease_key(stripped_fd)))
    if not r_12_pred:
        for pe in parse_disease_entities(stripped_fd):
            if (cid_base, pe) in pair_index:
                r_12_pred = pair_index[(cid_base, pe)]
                break
        if not r_12_pred:
            for ann in all_anns:
                ann_d_norm = normalize_disease_key(ann.get("target_disease", ""))
                s_norm = normalize_disease_key(stripped_fd)
                if ann_d_norm and s_norm and len(s_norm) >= 4 and (ann_d_norm in s_norm or s_norm in ann_d_norm):
                    r_12_pred = ann
                    break

    label_hyp = r_12_hyp.get("step12_dataset_label", cl) if r_12_hyp else cl
    role_hyp = r_12_hyp.get("step8_clinical_role", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    ev_str_hyp = r_12_hyp.get("step9_evidence_strength", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    rationale_hyp = r_12_hyp.get("rationale", "") if r_12_hyp else ""

    label_pred = r_12_pred.get("step12_dataset_label", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    role_pred = r_12_pred.get("step8_clinical_role", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    ev_str_pred = r_12_pred.get("step9_evidence_strength", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    rationale_pred = r_12_pred.get("rationale", "") if r_12_pred else ""

    all_hyps_block = ""
    if all_anns:
        hyp_lines = []
        for ann in all_anns:
            d_name = ann.get("target_disease", "UNKNOWN")
            d_lbl = ann.get("step12_dataset_label", "UNKNOWN")
            d_role = ann.get("step8_clinical_role", "UNKNOWN")
            d_str = ann.get("step9_evidence_strength", "UNKNOWN")
            hyp_lines.append(f"  * {d_name}: {d_lbl} (Role: {d_role}, Strength: {d_str})")
        all_hyps_block = "\n".join(hyp_lines)
    else:
        all_hyps_block = f"  * {td}: {cl}"

    lines = []
    lines.append(f'<case id="{cid}" index="{idx}">')
    lines.append("<case_metadata>")
    lines.append(f"- **Patient Case ID:** `{cid_base}`")
    lines.append(f"- **Target Hypothesis Audited by Pipeline:** `{td}`")
    lines.append(f"- **Ground Truth Annotation Label:** `{cl}`")
    lines.append(f"- **Method X (MedCase GPT-OSS 120B) Stripped Predicted Diagnosis:** **`{stripped_fd}`**")
    lines.append(f"- **Method X Closed-Set Taxonomy Scope Flag:** `{scope_flag}` (Raw Output: `{fd_raw}`)")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Verdict for `{td}`:** `{label_hyp}` (Role: `{role_hyp}`, Strength: `{ev_str_hyp}`)")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Verdict for Stripped Entity `{stripped_fd}`:** `{label_pred}` (Role: `{role_pred}`, Strength: `{ev_str_pred}`)")
    lines.append("</case_metadata>")
    lines.append("")
    lines.append("<all_12step_hypotheses_evaluated>")
    lines.append(all_hyps_block)
    lines.append("</all_12step_hypotheses_evaluated>")
    lines.append("")
    lines.append("<clinical_case_presentation source=\"raw_case_text\">")
    lines.append(raw_case_text)
    lines.append("</clinical_case_presentation>")
    lines.append("")
    lines.append("<method_x_reasoning>")
    lines.append(f"Primary Diagnosis Selected (Stripped): {stripped_fd}")
    lines.append(f"Closed-Set Scope Flag: {scope_flag} (Raw: {fd_raw})")
    lines.append("Breakpoint & Scope Analysis:")
    lines.append(analysis)
    lines.append("Differential Diagnostic Reasoning:")
    lines.append(reasoning_mc)
    lines.append("</method_x_reasoning>")
    lines.append("")
    lines.append("<method_y_grounding>")
    lines.append(f"- Adjudication for `{td}`: {label_hyp} (Role: {role_hyp}, Evidence: {ev_str_hyp})")
    lines.append(f"- Adjudication for Stripped Diagnosis `{stripped_fd}`: {label_pred} (Role: {role_pred}, Evidence: {ev_str_pred})")
    lines.append("- Direct Evidence Extracted for Hypothesis:")
    lines.append(format_evidence_block(r_12_hyp))
    if r_12_pred and r_12_pred != r_12_hyp:
        lines.append(f"- Direct Evidence Extracted for Stripped Entity (`{stripped_fd}`):")
        lines.append(format_evidence_block(r_12_pred))
    lines.append("- Clinical Rationale:")
    lines.append(rationale_hyp or rationale_pred or "Evidence evaluated against 12-step protocol.")
    lines.append("</method_y_grounding>")
    lines.append("")
    lines.append("</case>")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


def generate_claude_group2_prompt(
    group2: List[Any],
    pair_index: Dict[Tuple[str, str], Dict[str, Any]],
    all_by_case: Dict[str, List[Dict[str, Any]]],
    raw_text_map: Dict[str, str],
    split_name: str,
    batch_size: int = 5
) -> str:
    """Generates the complete Claude evaluation markdown benchmark file for Group 2 cases,
    chunked into copyable user prompt blocks of `batch_size` (default 5) cases per block.
    """
    n_cases = len(group2)
    total_batches = (n_cases + batch_size - 1) // batch_size if n_cases else 0
    lines = []
    lines.append(f"# Claude Forensic Evaluation Benchmark: Group 2 Stripped OUT_OF_SCOPE Audit ({split_name})")
    lines.append("")
    lines.append("> **Anthropic Claude Best-Practice Evaluation Benchmark (Group 2 Extension)**")
    lines.append(f"> Designed for forensic adjudication of the **{n_cases} Group 2 (`OUT_OF_SCOPE`) cases** in split `{split_name}`, partitioned into **{total_batches} copy-ready User Prompt blocks** ({batch_size} cases per block).")
    lines.append("> - **Method X:** Stanford MedCaseReasoning (GPT-OSS 120B) — Evaluated via its **stripped disease name** alongside its `OUT_OF_SCOPE` scope decision.")
    lines.append("> - **Method Y:** MultiCaRe 12-Step Clinical Annotation (Evidentiary verification protocol).")
    lines.append("> - **Enhancements:** Full verbatim `raw_case_text`, **Archetype D** (`BOTH_RIGHT` when both answers are valid), and **Dimension $D_6$ (Diagnostic Primacy)** grounded in the Spanish ICD-10 Coding Manual.")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("## 1. SYSTEM PROMPT (Copy into Claude 'System' field)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")
    lines.append("```xml")
    lines.append(build_claude_group2_system_prompt())
    lines.append("```")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append(f"## 2. USER PROMPT BATCHES ({total_batches} Blocks × up to {batch_size} Cases — Copy 1 Block per Chat)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")
    for b_idx in range(total_batches):
        start_pos = b_idx * batch_size
        end_pos = min(start_pos + batch_size, n_cases)
        batch_slice = group2[start_pos:end_pos]
        start_case_num = start_pos + 1
        end_case_num = end_pos
        batch_num = b_idx + 1
        batch_case_ids = ", ".join(f"`{r.get('case_id', '')}`" for r, _ in batch_slice)

        lines.append(f"### User Prompt Block {batch_num} of {total_batches} (Cases {start_case_num}–{end_case_num} of {n_cases})")
        lines.append(f"> **Cases in this block ({len(batch_slice)}):** {batch_case_ids}")
        lines.append("")
        lines.append("```xml")
        lines.append(
            build_claude_group2_user_header(
                len(batch_slice), batch_num, total_batches, start_case_num, end_case_num, n_cases, split_name
            )
        )
        lines.append("")
        lines.append(f"<!-- BATCH {batch_num} OF {total_batches}: CASES {start_case_num} TO {end_case_num} -->")
        lines.append("")
        for offset, (r_mc, _) in enumerate(batch_slice):
            global_idx = start_case_num + offset
            lines.append(format_claude_group2_case_entry(global_idx, r_mc, pair_index, all_by_case, raw_text_map))
        lines.append("</evaluation_task>")
        lines.append("```")
        lines.append("")
        lines.append("---")
        lines.append("")
    return "\n".join(lines)

## 11. Group 2 DeepSeek (R1 / DeepThink) Forensic Judicial Prompt Generator

This generator mirrors Section 6 (Group 3 DeepSeek Prompt Generator) with the four Group 2 enhancements:
- **Stripped `OUT_OF_SCOPE` Diagnosis + Scope Audit**
- **Full Verbatim `raw_case_text`**
- **Archetype D (`ARCHETYPE_D` — Dual-System Validity / `BOTH_RIGHT`)**
- **Dimension 6 ($D_6$: `diagnostic_primacy`, total `/30`)** grounded in the Spanish ICD-10 Coding Manual (*Manual de Codificación CIE-10-ES Diagnósticos*).

In [12]:
def build_deepseek_group2_header(n_cases: int, split_name: str) -> str:
    """Constructs DeepSeek judicial header for Group 2 with Archetype D and Dimension D6 (Spanish ICD-10 Coding Manual)."""
    closed_21_str = ", ".join(CLOSED_21_DISEASES)
    return f"""# DeepSeek Clinical Adjudication Benchmark: {n_cases} Group 2 Stripped OUT_OF_SCOPE Cases ({split_name})

> **INSTRUCTION FOR DEEPSEEK (DEEPTHINK / R1):**
> You are an expert Chief Medical Officer, Infectious Disease Specialist, ICD-10 Clinical Coding Authority, and Master Forensic Arbiter.
> Below are **{n_cases} REAL CLINICAL CASES** from Group 2 (`OUT_OF_SCOPE`) comparing:
> - **Method X:** Stanford MedCaseReasoning (GPT-OSS 120B) — Originally flagged `-> OUT_OF_SCOPE` against its closed 21-disease list ({closed_21_str}). We stripped `-> OUT_OF_SCOPE` so you can directly evaluate its **predicted disease name** alongside its scope flag.
> - **Method Y:** MultiCaRe 12-Step Clinical Annotation — Rigorous targeted evidentiary verification engine.
>
> Conduct an impartial, rigorous clinical and microbiological forensic audit using the full verbatim case text (`raw_case_text`).

---

## Forensic Evaluation Rubric (Rate 1 to 5 for each method across 6 Dimensions; Total /30)

1. **Diagnostic Accuracy & Consensus Alignment ($D_1$, 1–5):** (Singhal et al., Nature 2023; Tu et al., Nature 2025) Does the method identify a true etiologic or comorbid pathology supported by confirmatory laboratory/histopathological standards (E1/E2)?
2. **Rule-Out Rigor vs False-Positive Rule-In ($D_2$, 1–5):** (Sackett et al., 2000; Schiff et al., DEER Class 2.4, 2009) Does the method properly respect definitive negative laboratory tests (e.g. negative blood smears, negative biopsies), or does it commit confirmation bias?
3. **Temporality & Discourse Context Parsing ($D_3$, 1–5):** (Uzuner et al., JAMIA 2011; Harkema et al., 2009; DEER Class 3.2) Does the method distinguish between active current illness vs resolved past medical history (F4) or prophylactic vaccination (F5)?
4. **Evidence Grounding & Verbatim Fidelity ($D_4$, 1–5):** (Goyal et al. / CliniFact, JAMIA 2025) Are claims strictly anchored in objective clinical findings from the full `raw_case_text` without hallucinated clinical assumptions?
5. **Patient Safety & Diagnostic Prudence ($D_5$, 1–5):** (Singh & Graber, Diagnosis 2014) Would acting on this method's diagnosis protect the patient, or lead to toxic/unnecessary therapies?
6. **Diagnostic Primacy ($D_6$, 1–5):** (Grounded in the **Spanish ICD-10 Coding Manual / *Manual de Codificación CIE-10-ES Diagnósticos***) Two methods can both name real diseases present in the patient, yet only one may capture the centrally discussed / definitive principal diagnosis of the case:
   - **5:** Predicted label is the principal / definitive diagnosis that the source case identifies as the main disease (e.g., GLA over aspergillosis; ABPM over aspergillosis alone; IgA-PIGN over aspergillosis).
   - **3:** Predicted label is a real but secondary / comorbid / complicating condition that is explicitly present but not the principal diagnosis (e.g., calling the rheumatoid arthritis case “aspergillosis” when aspergilloma is past history; calling the CVID case “aspergillosis” when the admission is voriconazole photosensitivity).
   - **1:** Predicted label is unrelated, incorrect, or a trivial colonisation/contaminant presented as the main disease.

---

## Diagnostic Dilemma Archetypes & Pitfalls (DEER Cognitive Taxonomy + Dual-Validity Archetype D)
- **Archetype A (Rule-In vs Rule-Out / Temporality - DEER Class 3.2):** Inactive remote past history (F4) or affirmation despite negative confirmatory testing (F6).
- **Archetype B (Context / Prophylaxis / Presumptive - DEER Class 2.4 & 3.2):** Prophylactic vaccination or screening mistaken for active disease (F5), or presumptive treatment without proof.
- **Archetype C (Syndromic Mimic / Scope Error - DEER Class 3.2 & 4.1):** True clinical mimicry, multi-condition overlap, or false `OUT_OF_SCOPE` refusal when the stripped diagnosis is actually one of the 21 in-scope diseases.
- **Archetype D (Dual-System Validity — Both Systems Right):** Both systems provide clinically valid answers for the patient (e.g., Method X identifies the principal out-of-scope diagnosis or acute drug reaction, while Method Y accurately confirms a true comorbid/secondary target infection). **If Archetype D applies, mark the verdict as `BOTH_RIGHT` and use Dimension $D_6$ (Diagnostic Primacy) to reward the method that captured the centrally discussed principal diagnosis (5 vs 3).**

---

## Required Output Format for Each Case

<required_output_template>
### Adjudication Case [Case #]: [Case ID]

#### 1. Clinical Forensic Scratchpad
- **True Principal Pathology vs. Comorbidities:** [Principal diagnosis and secondary/comorbid conditions in raw_case_text]
- **Method X Scope Flag Audit:** [VALID_OUT_OF_SCOPE | FALSE_OUT_OF_SCOPE_REFUSAL]
- **Archetype Classification:** [Archetype A | Archetype B | Archetype C | Archetype D]
- **Diagnostic Traps Identified:** [Method X: None / F1-F6; Method Y: None / F1-F6]
- **Laboratory & Microbiological Reality:** [What objective tests actually showed]

#### 2. Comparative Scorecard
| Evaluation Dimension | Method X (MedCase Stripped) | Method Y (12-Step) | Forensic Comment |
|:---|:---:|:---:|:---|
| 1. Diagnostic Accuracy & Specificity ($D_1$) | [1-5] | [1-5] | [Brief critique] |
| 2. Rule-Out Rigor vs False Rule-In ($D_2$) | [1-5] | [1-5] | [Brief critique] |
| 3. Temporality & Discourse Context ($D_3$) | [1-5] | [1-5] | [Brief critique] |
| 4. Evidence Grounding & Fidelity ($D_4$) | [1-5] | [1-5] | [Brief critique] |
| 5. Patient Safety & Prudence ($D_5$) | [1-5] | [1-5] | [Brief critique] |
| 6. Diagnostic Primacy ($D_6$, Spanish ICD-10) | [1-5] | [1-5] | [Principal (5) vs Secondary/Comorbid (3) vs Unrelated/Colonization (1)] |
| **OVERALL TOTAL** | **/30** | **/30** | |

#### 3. Clinical Forensic Critique
- **Method X (Stripped Diagnosis & Scope) Analysis:** [Detailed breakdown of MedCase's stripped diagnosis and OUT_OF_SCOPE flag]
- **Method Y (12-Step Hypothesis) Analysis:** [Detailed breakdown of 12-Step's evidentiary grounding and primacy]

#### 4. Final Judicial Verdict
- **Adjudicated Winner:** **[METHOD_X | METHOD_Y | BOTH_RIGHT | NEITHER_VALID]**
- **Diagnostic Primacy Holder ($D_6$):** **[METHOD_X | METHOD_Y | BOTH | NEITHER]**
- **Definitive Medical Rationale:** [Clear, authoritative evidence-based justification]
- **Prevented Clinical Harm / Complementarity:** [Explanation of prevented harm or how both diagnoses complement clinical care]
</required_output_template>
"""


def build_deepseek_group2_batch_user_header(
    batch_cases_count: int,
    batch_num: int,
    total_batches: int,
    start_idx: int,
    end_idx: int,
    total_split_cases: int,
    split_name: str
) -> str:
    """Constructs a self-contained header for a single 5-case DeepSeek Group 2 batch block."""
    closed_21_str = ", ".join(CLOSED_21_DISEASES)
    return f"""# DeepSeek Clinical Adjudication Task — Batch {batch_num} of {total_batches} (Cases {start_idx}–{end_idx} of {total_split_cases} in `{split_name}`)

You are an expert Chief Medical Officer, Infectious Disease Specialist, ICD-10 Clinical Coding Authority, and Master Forensic Arbiter.
Adjudicate the **{batch_cases_count} Group 2 (`OUT_OF_SCOPE`) clinical cases** below (Cases {start_idx}–{end_idx}) comparing:
- **Method X (Stanford MedCaseReasoning, GPT-OSS 120B):** Originally appended `-> OUT_OF_SCOPE` against its closed 21-disease list ({closed_21_str}). We stripped `-> OUT_OF_SCOPE` so you can evaluate its **predicted disease name** directly alongside its scope flag.
- **Method Y (MultiCaRe 12-Step Clinical Annotation):** Targeted evidentiary verification engine.

For each of the {batch_cases_count} cases:
1. Audit both methods against the full verbatim clinical case report (`raw_case_text`).
2. Score both methods (1–5 scale, total `/30`) across all 6 dimensions:
   - **$D_1$ Diagnostic Accuracy** (confirmatory microbiology/histopathology E1/E2)
   - **$D_2$ Rule-Out Rigor** (respecting negative lab/biopsy findings)
   - **$D_3$ Temporality & Context** (acute vs remote cured history F4 or vaccination/prophylaxis F5)
   - **$D_4$ Evidence Grounding** (verbatim fidelity to `raw_case_text`)
   - **$D_5$ Patient Safety** (avoiding toxic unneeded therapy or missed lethal disease)
   - **$D_6$ Diagnostic Primacy** (Spanish ICD-10 Coding Manual / *Manual de Codificación CIE-10-ES Diagnósticos*): **5** = principal/definitive diagnosis centrally discussed as the main disease (e.g., GLA over aspergillosis; ABPM over aspergillosis alone; IgA-PIGN over aspergillosis); **3** = real but secondary/comorbid/complicating condition explicitly present (e.g., calling rheumatoid arthritis case 'aspergillosis' when aspergilloma is past history; calling CVID case 'aspergillosis' when admission is voriconazole photosensitivity); **1** = unrelated, incorrect, or trivial colonization/contaminant.
3. Classify the dilemma archetype: **Archetype A** (Rule-in vs Rule-out / Temporality), **Archetype B** (Context / Prophylaxis / Presumptive), **Archetype C** (Syndromic mimic / False `OUT_OF_SCOPE` refusal on an in-scope disease), or **Archetype D** (Both systems provide valid answers for the patient — mark Adjudicated Winner as **`BOTH_RIGHT`** and identify the Diagnostic Primacy Holder via $D_6$).
4. Output the 4-part adjudication per case: (1) Clinical Forensic Scratchpad, (2) Comparative Scorecard Table ($D_1 \to D_6$, `/30`), (3) Clinical Forensic Critique, and (4) Final Judicial Verdict (`METHOD_X | METHOD_Y | BOTH_RIGHT | NEITHER_VALID` + Diagnostic Primacy Holder)."""


def format_deepseek_group2_case_entry(
    idx: int,
    r_mc: Dict[str, Any],
    pair_index: Dict[Tuple[str, str], Dict[str, Any]],
    all_anns: List[Dict[str, Any]],
    raw_text_map: Dict[str, str]
) -> str:
    """Formats a single Group 2 case for DeepSeek evaluation using stripped OUT_OF_SCOPE diagnosis and full raw_case_text."""
    cid = r_mc.get("case_id", "")
    cid_base = cid.split("__")[0].strip()
    td = r_mc.get("target_disease", "")
    cl = r_mc.get("clinical_label", "")
    fd_raw = r_mc.get("final_diagnosis", "").strip()
    stripped_fd, scope_flag = extract_stripped_diagnosis(fd_raw)
    raw_case_text = resolve_raw_case_text(r_mc, raw_text_map)
    analysis = r_mc.get("case_analysis", "").strip()
    reasoning_mc = r_mc.get("diagnostic_reasoning", "").strip()

    r_12_hyp = pair_index.get((cid_base, normalize_disease_key(td)))

    # Resolve stripped predicted entity in 12-step annotations if evaluated
    r_12_pred = pair_index.get((cid_base, normalize_disease_key(stripped_fd)))
    if not r_12_pred:
        for pe in parse_disease_entities(stripped_fd):
            if (cid_base, pe) in pair_index:
                r_12_pred = pair_index[(cid_base, pe)]
                break
        if not r_12_pred:
            for ann in all_anns:
                ann_d_norm = normalize_disease_key(ann.get("target_disease", ""))
                s_norm = normalize_disease_key(stripped_fd)
                if ann_d_norm and s_norm and len(s_norm) >= 4 and (ann_d_norm in s_norm or s_norm in ann_d_norm):
                    r_12_pred = ann
                    break

    label_hyp = r_12_hyp.get("step12_dataset_label", cl) if r_12_hyp else cl
    role_hyp = r_12_hyp.get("step8_clinical_role", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    ev_str_hyp = r_12_hyp.get("step9_evidence_strength", "UNKNOWN") if r_12_hyp else "UNKNOWN"
    rationale_hyp = r_12_hyp.get("rationale", "") if r_12_hyp else ""

    label_pred = r_12_pred.get("step12_dataset_label", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    role_pred = r_12_pred.get("step8_clinical_role", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    ev_str_pred = r_12_pred.get("step9_evidence_strength", "NOT_IN_29_TARGET_ANNOTATIONS") if r_12_pred else "NOT_IN_29_TARGET_ANNOTATIONS"
    rationale_pred = r_12_pred.get("rationale", "") if r_12_pred else ""

    lines = []
    lines.append(f"### Case {idx}: `{cid}`")
    lines.append("")
    lines.append(f"- **Target Hypothesis Audited:** `{td}` | **Ground Truth Label:** `{cl}`")
    lines.append(f"- **Method X (MedCase GPT-OSS 120B) Stripped Diagnosis:** **`{stripped_fd}`**")
    lines.append(f"- **Method X Closed-Set Scope Flag:** `{scope_flag}` (Raw Output: `{fd_raw}`)")
    lines.append(f"- **Method Y (MultiCaRe 12-Step) Status for `{td}`:** **`{label_hyp}`** (Role: `{role_hyp}`, Strength: `{ev_str_hyp}`)")
    lines.append(f"- **Method Y Status for Stripped Entity `{stripped_fd}`:** **`{label_pred}`** (Role: `{role_pred}`)")
    lines.append("")
    lines.append("#### 1. Full Verbatim Clinical Case Report (`raw_case_text`)")
    lines.append("> " + raw_case_text.replace("\n", "\n> "))
    lines.append("")
    lines.append("#### 2. Method X: Stanford MedCaseReasoning (GPT-OSS 120B Trajectory)")
    lines.append(f"- **Stripped Primary Diagnostic Choice:** **`{stripped_fd}`**")
    lines.append(f"- **Scope Flag Assigned:** `{scope_flag}` (`{fd_raw}`)")
    lines.append("- **Breakpoint & Scope Analysis:**")
    lines.append("<breakpoint_and_scope_analysis>")
    lines.append(analysis)
    lines.append("</breakpoint_and_scope_analysis>")
    lines.append("- **Differential Diagnostic Reasoning Steps:**")
    lines.append(reasoning_mc)
    lines.append("")
    lines.append("#### 3. Method Y: MultiCaRe 12-Step Clinical Annotation (Evidentiary Audit)")
    lines.append(f"- **Adjudication for `{td}`:** **`{label_hyp}`** (Role: `{role_hyp}` | Evidence: `{ev_str_hyp}`)")
    lines.append(f"- **Adjudication for Stripped Entity `{stripped_fd}`:** **`{label_pred}`** (Role: `{role_pred}` | Evidence: `{ev_str_pred}`)")
    lines.append("- **Direct Evidence Extracted for Hypothesis:**")
    lines.append(format_evidence_block(r_12_hyp))
    if r_12_pred and r_12_pred != r_12_hyp:
        lines.append(f"- **Direct Evidence Extracted for Stripped Entity (`{stripped_fd}`):**")
        lines.append(format_evidence_block(r_12_pred))
    lines.append("- **Clinical Rationale:**")
    lines.append(rationale_hyp or rationale_pred or "Evidence and diagnostic criteria evaluated against standard protocol.")
    lines.append("")
    lines.append("---")
    lines.append("")
    return "\n".join(lines)


def generate_deepseek_group2_prompt(
    group2: List[Any],
    pair_index: Dict[Tuple[str, str], Dict[str, Any]],
    all_by_case: Dict[str, List[Dict[str, Any]]],
    raw_text_map: Dict[str, str],
    split_name: str,
    batch_size: int = 5
) -> str:
    """Generates the complete DeepSeek evaluation markdown benchmark file for Group 2 cases,
    chunked into copyable user prompt blocks of `batch_size` (default 5) cases per block.
    """
    n_cases = len(group2)
    total_batches = (n_cases + batch_size - 1) // batch_size if n_cases else 0
    lines = []
    lines.append(f"# DeepSeek Forensic Evaluation Benchmark: Group 2 Stripped OUT_OF_SCOPE Audit ({split_name})")
    lines.append("")
    lines.append(f"> **DeepSeek (DeepThink / R1) Evaluation Benchmark — {n_cases} Cases across {total_batches} Copy-Ready Blocks ({batch_size} Cases/Block)**")
    lines.append("")
    lines.append(f"## Executive Summary Table ({n_cases} Group 2 Cases)")
    lines.append("")
    lines.append("| # | Batch | Case ID | Target Disease | 12-Step Label | Method X Stripped Diagnosis | Method X Scope Flag | Method Y Status for Stripped Entity |")
    lines.append("|:---:|:---:|:---|:---:|:---:|:---|:---:|:---|")

    for idx, (r_mc, all_anns) in enumerate(group2, 1):
        b_num = (idx - 1) // batch_size + 1
        cid = r_mc.get("case_id", "")
        cid_base = cid.split("__")[0].strip()
        td = r_mc.get("target_disease", "")
        cl = r_mc.get("clinical_label", "")
        fd_raw = r_mc.get("final_diagnosis", "").strip()
        stripped_fd, scope_flag = extract_stripped_diagnosis(fd_raw)

        r_12_pred = pair_index.get((cid_base, normalize_disease_key(stripped_fd)))
        if not r_12_pred:
            for pe in parse_disease_entities(stripped_fd):
                if (cid_base, pe) in pair_index:
                    r_12_pred = pair_index[(cid_base, pe)]
                    break
            if not r_12_pred:
                for ann in all_anns:
                    ann_d_norm = normalize_disease_key(ann.get("target_disease", ""))
                    s_norm = normalize_disease_key(stripped_fd)
                    if ann_d_norm and s_norm and len(s_norm) >= 4 and (ann_d_norm in s_norm or s_norm in ann_d_norm):
                        r_12_pred = ann
                        break

        lbl_pred = r_12_pred.get("step12_dataset_label", "OUTSIDE_29_TARGETS") if r_12_pred else "OUTSIDE_29_TARGETS"
        lines.append(f"| {idx} | B{b_num} | `{cid}` | {td} | **{cl}** | **`{stripped_fd}`** | `{scope_flag}` | `{lbl_pred}` |")

    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("## 1. MASTER SYSTEM / RUBRIC PROMPT (Optional Reference or First Message)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")
    lines.append("```markdown")
    lines.append(build_deepseek_group2_header(n_cases, split_name))
    lines.append("```")
    lines.append("")
    lines.append("---")
    lines.append("")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append(f"## 2. USER PROMPT BATCHES ({total_batches} Blocks × up to {batch_size} Cases — Copy 1 Block per Chat)")
    lines.append("## ═══════════════════════════════════════════════════")
    lines.append("")

    for b_idx in range(total_batches):
        start_pos = b_idx * batch_size
        end_pos = min(start_pos + batch_size, n_cases)
        batch_slice = group2[start_pos:end_pos]
        start_case_num = start_pos + 1
        end_case_num = end_pos
        batch_num = b_idx + 1
        batch_case_ids = ", ".join(f"`{r.get('case_id', '')}`" for r, _ in batch_slice)

        lines.append(f"### User Prompt Block {batch_num} of {total_batches} (Cases {start_case_num}–{end_case_num} of {n_cases})")
        lines.append(f"> **Cases in this block ({len(batch_slice)}):** {batch_case_ids}")
        lines.append("")
        lines.append("```markdown")
        lines.append(
            build_deepseek_group2_batch_user_header(
                len(batch_slice), batch_num, total_batches, start_case_num, end_case_num, n_cases, split_name
            )
        )
        lines.append("")
        lines.append("---")
        lines.append("")
        for offset, (r_mc, all_anns) in enumerate(batch_slice):
            global_idx = start_case_num + offset
            lines.append(format_deepseek_group2_case_entry(global_idx, r_mc, pair_index, all_anns, raw_text_map))
        lines.append("```")
        lines.append("")
        lines.append("---")
        lines.append("")

    return "\n".join(lines)

## 12. Group 2 Batch Prompt Orchestrator Across All 5 Splits

Executes Group 2 prompt generation across all 5 splits without overwriting Group 3 prompts, saving:
- `outputs/<split_name>/claude_group2_benchmark_prompt.md`
- `outputs/<split_name>/deepseek_group2_benchmark_prompt.md`

In [13]:
def run_group2_prompt_pipeline(
    mismatch_dir: Path,
    output_base_dir: Path,
    annotated_map: Dict[Tuple[str, str], Dict[str, Any]],
    annotated_by_case: Dict[str, List[Dict[str, Any]]],
    raw_text_map: Dict[str, str],
    batch_size: int = 5
) -> List[Dict[str, Any]]:
    """Generates Group 2 Claude and DeepSeek benchmark prompts across all 5 mismatch splits,
    partitioning cases into copy-ready user prompt blocks of `batch_size` (default 5) cases per block.
    """
    output_base_dir.mkdir(parents=True, exist_ok=True)
    mismatch_files = sorted(list(mismatch_dir.glob("*.jsonl")))
    if not mismatch_files:
        raise FileNotFoundError(f"No jsonl files found in mismatch directory: {mismatch_dir}")

    g2_manifest: List[Dict[str, Any]] = []
    total_g2_cases = 0
    total_g2_batches = 0

    for mf in mismatch_files:
        split_name = mf.stem
        split_out_dir = output_base_dir / split_name
        split_out_dir.mkdir(parents=True, exist_ok=True)

        with open(mf, "r", encoding="utf-8") as fp:
            records = [json.loads(line) for line in fp if line.strip()]

        _, group2, _ = partition_records(records, annotated_map, annotated_by_case)
        n_g2 = len(group2)
        n_batches = (n_g2 + batch_size - 1) // batch_size if n_g2 else 0
        total_g2_cases += n_g2
        total_g2_batches += n_batches

        claude_g2_path = split_out_dir / "claude_group2_benchmark_prompt.md"
        deepseek_g2_path = split_out_dir / "deepseek_group2_benchmark_prompt.md"

        if group2:
            claude_g2_md = generate_claude_group2_prompt(
                group2, annotated_map, annotated_by_case, raw_text_map, split_name, batch_size=batch_size
            )
            with open(claude_g2_path, "w", encoding="utf-8") as f:
                f.write(claude_g2_md)

            deepseek_g2_md = generate_deepseek_group2_prompt(
                group2, annotated_map, annotated_by_case, raw_text_map, split_name, batch_size=batch_size
            )
            with open(deepseek_g2_path, "w", encoding="utf-8") as f:
                f.write(deepseek_g2_md)

        print(f"  ✓ [{split_name}] {n_g2} Group 2 cases -> {n_batches} User Prompt Blocks ({batch_size} cases/block):")
        print(f"      - {claude_g2_path.relative_to(mismatch_dir)} ({claude_g2_path.stat().st_size:,} bytes)")
        print(f"      - {deepseek_g2_path.relative_to(mismatch_dir)} ({deepseek_g2_path.stat().st_size:,} bytes)")

        g2_manifest.append({
            "split_name": split_name,
            "group2_cases": n_g2,
            "user_prompt_blocks": n_batches,
            "batch_size": batch_size,
            "claude_prompt_path": str(claude_g2_path.relative_to(mismatch_dir)),
            "deepseek_prompt_path": str(deepseek_g2_path.relative_to(mismatch_dir)),
        })

    print(f"\n=======================================================")
    print(f"GROUP 2 PROMPT GENERATION COMPLETE! Total Cases: {total_g2_cases} across {total_g2_batches} User Prompt Blocks")
    print(f"=======================================================")
    return g2_manifest


group2_prompt_manifest = run_group2_prompt_pipeline(
    MISMATCH_DIR, OUTPUT_BASE_DIR, annotated_map, annotated_by_case, raw_case_text_map
)

  ✓ [unmatched_part_1_Khai] 54 Group 2 cases -> 11 User Prompt Blocks (5 cases/block):
      - outputs/unmatched_part_1_Khai/claude_group2_benchmark_prompt.md (531,471 bytes)
      - outputs/unmatched_part_1_Khai/deepseek_group2_benchmark_prompt.md (538,894 bytes)
  ✓ [unmatched_part_2_Khoi] 58 Group 2 cases -> 12 User Prompt Blocks (5 cases/block):
      - outputs/unmatched_part_2_Khoi/claude_group2_benchmark_prompt.md (599,227 bytes)
      - outputs/unmatched_part_2_Khoi/deepseek_group2_benchmark_prompt.md (605,148 bytes)
  ✓ [unmatched_part_3_AMinh] 67 Group 2 cases -> 14 User Prompt Blocks (5 cases/block):
      - outputs/unmatched_part_3_AMinh/claude_group2_benchmark_prompt.md (673,692 bytes)
      - outputs/unmatched_part_3_AMinh/deepseek_group2_benchmark_prompt.md (681,491 bytes)
  ✓ [unmatched_part_4_Khoa] 37 Group 2 cases -> 8 User Prompt Blocks (5 cases/block):
      - outputs/unmatched_part_4_Khoa/claude_group2_benchmark_prompt.md (335,404 bytes)
      - outputs/unmatched_pa